# Baseline 8 — FICR 정산 구간 안착 (카드 A~F 통합)

`baseline_7.ipynb`의 IDW 공간집계·SHAP-LightGBM·버전별 LSTM 파이프라인을 그대로 이어받아
`MD/baseline7_howto_next.md`의 카드 A~G를 반영한다. baseline_7 잔여 격차(Persistence oracle
대비 0.1294점)의 86%가 FICR에서 온다는 진단에 따라, 이 노트북의 목표는 NMAE를 더 낮추는 것이
아니라 **시간별 오차율이 정산단가 4원 구간(오차율 ≤6%)에 들어가는 비율(밴드 안착률)을
높이는 것**이다. 그룹별 검증 총점이 가장 높은 최종 후보(원본 / 사후보정 / FICR 목적함수 /
앙상블)를 다시 전체 학습해 `submission_baseline8.csv`까지 생성한다.

| Step | 내용 |
|------|------|
| 0 | 환경 설정 — 라이브러리·경로·시드 |
| 1 | 정상 IDW + 예보 변화량 피처 전처리 및 캐시 |
| 2 | LightGBM 전체/SHAP선별/FICR목적함수 학습 + 예측 저장소 (카드 A·C) |
| 3 | 동일 선별 피처로 LSTM v1/v2/v3 공정 비교 + 예측 저장소 (카드 A) |
| 4 | 공식 총점 집계 + 그룹별 최적 모델 선택 + 오차 밴드 진단 (카드 A) |
| 5 | 사후 보정 — 전역 (카드 B) |
| 6 | 지능형 HPO — GWO (카드 D) |
| 7 | 시각대 오차 진단 (카드 E) + 사후 보정 시각대 확장 (카드 B) |
| 8 | 최종 후보 채택 + 앙상블 (카드 F) |
| 9 | 전체 학습 + 평가 기간 예측 + 제출 검증 (카드 F 반영) |
| 부록 | VMD+mRMR 피처 스파이크 (카드 G, 선택·기본 비활성) |

### 통합 원칙

- `turbine_meta["group"]`은 `"kpx_group_N"`으로 맞추고 IDW 신호가 0이 아닌지 즉시 검사한다.
- 모든 모델은 같은 마지막 20% 검증 구간과 `SEQ_LEN=24` 이후 시각으로 평가한다.
- SHAP 선별은 학습 구간으로 학습한 모델과 검증 표본만 사용하며, LSTM의 결측 대체·스케일러도 학습 구간에만 적합한다.
- Persistence는 직전 실제값을 쓰는 oracle 참고선이므로 최종 모델 선택에서 제외한다.
- **모든 보정·FICR목적함수·HPO·앙상블 탐색은 검증 구간(뒤 20%)에서만 하고, 평가(2025년) 구간에는
  검증에서 고정한 파라미터를 그대로 적용만 한다.**
- **채택 기준은 항상 baseline_7 실측 그룹별 기준선**(그룹1 `0.6555` / 그룹2 `0.6877` / 그룹3
  `0.6583`)**이다.** 이를 넘지 못하면 그 그룹은 원안(`LGBM_selected`/`LSTM_*`)을 유지한다.


---
## Step 0. 환경 설정

LightGBM·SHAP·PyTorch가 없는 환경에서만 다음 설치 줄의 주석을 해제해 한 번 실행한다.
설치 후에는 커널을 재시작한다.


In [ ]:
# %pip install lightgbm shap torch scikit-learn
# 현재 커널에 패키지가 없을 때만 실행한다.


In [ ]:
import sys
from pathlib import Path

EXPECTED_VENV = "WINDFORCE/.venv"
# 이 노트북이 돌아야 하는 가상환경 경로 조각
if EXPECTED_VENV not in Path(sys.executable).as_posix():
    raise RuntimeError(
        f"잘못된 커널입니다: {sys.executable}\n"
        f"VS Code 우상단 커널 선택에서 WINDFORCE/.venv를 고르세요.\n"
        f"주피터랩이면 kernel.json의 argv[0]이 절대경로인지 확인하세요."
    )
    # 전역 파이썬으로 뜬 커널을 서드파티 임포트 전에 차단한다.
print(f"kernel: {sys.executable}")
# 어느 인터프리터로 돌았는지 실행 기록에 남긴다.

import os
import re
import time
import warnings
from typing import Optional

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
from sklearn.preprocessing import MinMaxScaler
# 데이터 분석·트리 모델·설명가능성·딥러닝 라이브러리

warnings.filterwarnings("ignore")
# 반복 실험 로그를 가리는 라이브러리 버전 경고를 억제한다.


In [ ]:
ROOT: str = next(
    (
        path
        for path in [
            "D:/workspaces/WINDFORCE",
            "d:/workspaces/WINDFORCE",
            "/Users/ksydata/WINDFORCE",
        ]
        if os.path.exists(path)
    ),
    os.getcwd(),
)
# Windows/macOS 실행 환경에서 실제로 존재하는 프로젝트 루트를 선택한다.
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
# Windforce 패키지 탐색 경로를 추가한다.

from Windforce import (
    EvaluationMetrics,
    GFSFeatureEngineer,
    LDAPSFeatureEngineer,
    RATED_CAPACITY_KW,
    TIME_STEP_HOURS,
    WindforceDataLoader,
)
# 데이터 로더·전처리기·공식 평가지표를 루트 패키지에서 가져온다.
from Windforce.Modeling import (
    AllFeaturesVariant,
    BaselineModels,
    WindforceDatasetBuilder,
    make_lstm_pipeline,
)
# baseline 7에서 검증한 모델링 구성요소를 재사용한다.

print(f"ROOT: {ROOT}")
print(f"lightgbm {lgb.__version__} / shap {shap.__version__} / torch {torch.__version__}")
# 실행 경로와 핵심 라이브러리 버전을 재현 로그에 남긴다.


In [ ]:
pd.set_option("display.max_columns", None)
# 데이터프레임의 모든 컬럼을 생략 없이 표시한다.
pd.options.display.float_format = "{:,.6f}".format
# 지수 표기 대신 소수점 6자리 실수로 표시한다.

GROUPS = [1, 2, 3]
# KPX 평가 그룹 번호 목록
SEQ_LEN = 24
# 최근 24시간의 기상 흐름으로 다음 시각 발전량(kWh)을 예측한다.
SEED = 42
# 모든 확률적 모델과 표본 추출에 쓰는 재현 시드
TEST_RATIO = 0.2
# 시간순 마지막 20%를 공통 검증 구간으로 사용한다.
SHAP_SAMPLE_SIZE = 2_000
# SHAP 계산 속도를 위한 그룹별 검증 표본 수
CORR_THRESHOLD = 0.95
# 이 값을 넘는 절대 상관 피처 쌍에서 SHAP 순위가 낮은 쪽을 제거한다.
NOISE_FEATURE_COUNT = 3
# null importance 기준선을 만들 무작위 정규분포 피처 개수
CUMULATIVE_RATIO = 1.0
# baseline 5 HOWTO 실측 최적값으로, 노이즈·상관 제거 뒤 꼬리 절단은 하지 않는다.
PREP_DIR = f"{ROOT}/prep"
# baseline 8 전처리 완료본을 저장하는 폴더

BASELINE7_SCORE = {1: 0.6555, 2: 0.6877, 3: 0.6583}
# baseline_7_results.csv 실측 그룹별 기준선(best_by_group 총점). 카드 B~F 채택 기준으로 쓴다.

os.makedirs(PREP_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(8)
# 12코어 전부를 쓰면 배치 64 구간에서 스레드 동기화 비용이 연산량을 넘어선다.
# numpy와 PyTorch 난수 시드를 고정한다.


---
## Step 1. 정상 IDW + 예보 변화량 피처 전처리 및 캐시

```text
LDAPS 16격자 / GFS 9격자
            ↓ 물리 파생변수
turbine_meta(group 문자열·위경도·용량) 기반 IDW
            ↓
KPX 그룹 × 시각 공간가중 피처
            ↓ transformForecastFeature()
1h/3h/6h 변화량·램프 피처
            ↓
prep/baseline8_dataset_{train|test}_groupN.csv.gz
```

캐시는 baseline 8 전용 파일명을 써서 이전 노트북의 전처리 결과와 섞이지 않게 한다.
피처 엔지니어링 로직 자체는 baseline_7과 동일하므로(카드 G를 켜지 않는 한 변경 없음),
`Windforce` 패키지의 전처리기·IDW 유틸을 그대로 재사용한다.


In [ ]:
def parseDMS(coord_str: str) -> tuple[float, float]:
    """도분초(DMS) 좌표를 십진수(DD) 위도·경도로 변환하는 함수

    Args:
        - coord_str: `37°16'55.61"N 128°57'02.10"E` 형식 문자열

    Returns:
        - (위도, 경도) 십진수 튜플 (도 단위)
    """
    tokens = re.findall(r"[\d.]+|[NSEW]", coord_str.replace('""', '"'))
    # 숫자와 방향(N/S/E/W)을 순서대로 분리한다.
    if len(tokens) != 8:
        raise ValueError(f"지원하지 않는 DMS 좌표 형식입니다: {coord_str}")
    # 위도·경도 각각 도/분/초/방향 4개 토큰인지 검증한다.

    lat = float(tokens[0]) + float(tokens[1]) / 60.0 + float(tokens[2]) / 3_600.0
    lon = float(tokens[4]) + float(tokens[5]) / 60.0 + float(tokens[6]) / 3_600.0
    # 도 + 분/60 + 초/3,600으로 십진도 단위로 환산한다.
    if tokens[3] == "S":
        lat = -lat
    if tokens[7] == "W":
        lon = -lon
    # 남반구(S)와 서경(W)은 음수 좌표로 변환한다.
    return lat, lon
    # IDW 거리 계산에 사용할 (위도, 경도) 튜플을 반환한다.


def buildTurbineMeta(kpx_info: pd.DataFrame) -> pd.DataFrame:
    """KPX 터빈 메타를 IDW 공간집계 계약에 맞게 변환하는 함수

    Args:
        - kpx_info: 터빈 1기 = 1행인 원본 메타 DataFrame

    Returns:
        - group / lat / lon / cap_kw 컬럼을 가진 DataFrame
    """
    kpx = kpx_info.copy()
    # 호출자가 보관한 원본 DataFrame을 변형하지 않는다.
    coords = kpx["좌표(Google)"].apply(parseDMS)
    kpx["lat"] = coords.apply(lambda value: value[0])
    kpx["lon"] = coords.apply(lambda value: value[1])
    # DMS 좌표를 십진 위도·경도(도 단위)로 분리한다.

    meta = kpx[["KPX그룹", "lat", "lon", "설비용량(MW)"]].rename(
        columns={"KPX그룹": "group", "설비용량(MW)": "cap_kw"}
    ).copy()
    meta["cap_kw"] = meta["cap_kw"] * 1_000.0
    # 설비용량을 MW에서 kW로 즉시 환산한다.
    meta["group"] = "kpx_group_" + meta["group"].astype(int).astype(str)
    # IDW 공통 계약과 같은 문자열 키로 맞춰 0 가중치 행렬 재발을 차단한다.
    return meta.reset_index(drop=True)
    # 행렬 인덱스와 터빈 행 순서를 일치시켜 반환한다.


In [ ]:
class PreprocessedDatasetBuilder:
    """LDAPS·GFS 원본을 baseline 8 그룹별 학습·평가 캐시로 만드는 클래스

    사용 순서:
        builder = PreprocessedDatasetBuilder(loader, PREP_DIR)
        builder.buildAll()
        train_df = builder.load(1, "train")
    """

    def __init__(
        self,
        loader: WindforceDataLoader,
        prep_dir: str,
        groups: list[int] = GROUPS,
    ):
        self.loader = loader
        # 원본 CSV를 지연 로드·캐시하는 데이터 로더
        self.prep_dir = prep_dir
        # gzip 전처리 완료본 저장 폴더
        self.groups = groups
        # 처리할 KPX 그룹 번호 목록
        self.turbine_meta: Optional[pd.DataFrame] = None
        # buildAll()에서 생성하는 IDW용 터빈 메타

    def _path(self, group: int, split: str) -> str:
        """그룹·데이터 구간별 baseline 8 캐시 경로를 만드는 내부 메서드"""
        return f"{self.prep_dir}/baseline8_dataset_{split}_group{group}.csv.gz"
        # pyarrow 추가 의존성 없이 압축 가능한 gzip CSV 경로를 반환한다.

    def isCached(self) -> bool:
        """모든 그룹의 학습·평가 캐시가 존재하는지 확인하는 메서드"""
        return all(
            os.path.exists(self._path(group, split))
            for group in self.groups
            for split in ("train", "test")
        )
        # 파일 하나라도 빠지면 전체 train/test를 같은 코드 버전으로 다시 만든다.

    def _transformSplit(
        self,
        ldaps_key: str,
        gfs_key: str,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        """LDAPS·GFS 한 구간을 그룹 단위 예보 피처로 변환하는 내부 메서드"""
        ldaps_fe = LDAPSFeatureEngineer()
        gfs_fe = GFSFeatureEngineer()
        # 전체 격자를 처리한 뒤 같은 터빈 메타로 세 그룹에 공간 매핑한다.

        ldaps_grid = ldaps_fe.transformLDAPS(self.loader[ldaps_key])
        ldaps_group = ldaps_fe.transformToGroupIDW(ldaps_grid, self.turbine_meta)
        ldaps_group = ldaps_fe.transformForecastFeature(ldaps_group)
        # LDAPS 물리 피처 → IDW → 그룹별 1h/3h/6h 변화량 순서로 처리한다.

        gfs_grid = gfs_fe.transformGFS(self.loader[gfs_key])
        gfs_group = gfs_fe.transformToGroupIDW(gfs_grid, self.turbine_meta)
        gfs_group = gfs_fe.transformForecastFeature(gfs_group)
        # GFS 물리 피처 → IDW → 허브풍속 기반 변화량 순서로 처리한다.

        print(f"  LDAPS: 격자 {ldaps_grid.shape} → 그룹 {ldaps_group.shape}")
        print(f"  GFS  : 격자 {gfs_grid.shape} → 그룹 {gfs_group.shape}")
        # 원본 격자와 그룹 집계 shape를 실행 로그에 남긴다.
        return ldaps_group, gfs_group
        # 한 행 = 한 그룹 × 한 시각인 두 예보표를 반환한다.

    @staticmethod
    def _checkSignal(ldaps_group: pd.DataFrame, gfs_group: pd.DataFrame) -> None:
        """IDW 결과의 대표 풍속 신호가 0 행렬이 아닌지 검사하는 내부 메서드"""
        checks = {
            "LDAPS ws10": (ldaps_group, "ws10"),
            "GFS gfs_ws_hub": (gfs_group, "gfs_ws_hub"),
        }
        # 각 기상원에서 발전량과 직접 연결되는 대표 풍속(m/s)을 검사한다.
        for name, (df, column) in checks.items():
            if column not in df.columns or float(df[column].abs().max()) <= 0.0:
                raise ValueError(f"{name} IDW 신호가 없거나 전부 0입니다")
            # 그룹 키나 공간가중 버그가 조용히 학습 단계로 넘어가지 않게 즉시 실패한다.

    def buildAll(self, force: bool = False) -> None:
        """전체 그룹의 baseline 8 학습·평가 캐시를 생성하는 메서드

        Args:
            - force: True면 기존 baseline 8 캐시를 무시하고 다시 만든다.
        """
        if self.isCached() and not force:
            print("✅ baseline 8 전처리 캐시가 있어 재사용합니다.")
            return
            # 같은 접두사의 6개 캐시가 모두 있으면 약 1분의 전처리를 건너뛴다.

        started = time.time()
        self.turbine_meta = buildTurbineMeta(self.loader["kpx_info"])
        # 정규화된 그룹 키·위경도·설비용량(kW)을 한 번만 만든다.

        print("[train] 격자 전처리 + IDW 그룹 집계 중...")
        ldaps_train, gfs_train = self._transformSplit("ldaps_train", "gfs_train")
        print("[test] 격자 전처리 + IDW 그룹 집계 중...")
        ldaps_test, gfs_test = self._transformSplit("ldaps_test", "gfs_test")
        self._checkSignal(ldaps_train, gfs_train)
        self._checkSignal(ldaps_test, gfs_test)
        # 학습·평가 양쪽에서 대표 풍속 피처가 실제 값인지 확인한다.

        train_builder = WindforceDatasetBuilder(
            ldaps_train,
            gfs_train,
            self.loader["train_labels"],
        )
        # 학습 예보 피처에 그룹별 실제 발전량(kWh)을 시각 기준으로 결합한다.

        test_times = pd.to_datetime(pd.Series(ldaps_test["forecast_kst_dtm"].unique()))
        dummy_labels = pd.DataFrame({"kst_dtm": test_times})
        for group in self.groups:
            dummy_labels[f"kpx_group_{group}"] = 0.0
        # 평가 기간은 라벨이 없으므로 같은 빌더 경로를 쓰기 위한 0 kWh 더미를 만든다.
        test_builder = WindforceDatasetBuilder(ldaps_test, gfs_test, dummy_labels)
        # 학습·평가 피처 병합 규칙과 컬럼 접미사를 완전히 동일하게 유지한다.

        for group in self.groups:
            for split, builder in (("train", train_builder), ("test", test_builder)):
                table = builder.build(group)
                path = self._path(group, split)
                table.to_csv(
                    path,
                    index=False,
                    encoding="utf-8-sig",
                    compression="gzip",
                )
                # UTF-8 BOM·gzip 형식으로 저장해 한글 호환성과 용량을 함께 지킨다.
                print(f"✅ [그룹{group}] {split}: {table.shape} → {os.path.basename(path)}")
        print(f"전처리 완료: {time.time() - started:.1f}초")
        # 전체 캐시 생성 시간을 재현 로그에 남긴다.

    def load(self, group: int, split: str = "train") -> pd.DataFrame:
        """저장한 그룹 캐시를 읽어 시간순으로 반환하는 메서드"""
        path = self._path(group, split)
        if not os.path.exists(path):
            raise FileNotFoundError(f"baseline 8 전처리 캐시가 없습니다: {path}")
        # buildAll() 실행 누락을 명확한 파일 경로와 함께 알린다.
        df = pd.read_csv(path, encoding="utf-8-sig")
        df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
        # CSV 왕복에서 문자열이 된 예보 대상 시각을 datetime으로 복원한다.
        return df.sort_values("forecast_kst_dtm").reset_index(drop=True)
        # 시계열 분할과 제출 재색인이 안전하도록 정렬 상태로 반환한다.


In [ ]:
# 데이터 경로를 확인하고 baseline 8 전처리 캐시를 준비한다.
loader = WindforceDataLoader(root=ROOT)
path_status = loader.check_paths()
if not all(path_status.values()):
    missing = [name for name, exists in path_status.items() if not exists]
    raise FileNotFoundError(f"필수 입력 파일이 없습니다: {missing}")
# 일부 파일이 없는 상태로 긴 전처리를 시작하지 않게 막는다.

prep_builder = PreprocessedDatasetBuilder(loader, PREP_DIR)
prep_builder.buildAll()
# 캐시가 없을 때만 정상 IDW·램프 피처 전처리를 실행한다.


In [ ]:
def splitByTime(
    df: pd.DataFrame,
    target_col: str,
    feature_cols: list[str],
    test_ratio: float = TEST_RATIO,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray]:
    """그룹 데이터를 학습·검증 구간으로 시간순 분할하는 함수

    Returns:
        - X_train, X_valid, y_train, y_valid 튜플
    """
    split = int(len(df) * (1.0 - test_ratio))
    # 앞 80%를 학습, 뒤 20%를 미래 검증 구간으로 고정한다.
    X_train = df.iloc[:split][feature_cols].reset_index(drop=True)
    X_valid = df.iloc[split:][feature_cols].reset_index(drop=True)
    y_train = df.iloc[:split][target_col].to_numpy(dtype=float)
    y_valid = df.iloc[split:][target_col].to_numpy(dtype=float)
    # 피처와 발전량(kWh) 타깃을 같은 행 경계로 잘라 인덱스를 맞춘다.
    return X_train, X_valid, y_train, y_valid
    # 랜덤 셔플 없이 시간 순서를 보존한 네 객체를 반환한다.


datasets: dict[int, dict] = {}
# 그룹별 전체 표·공통 피처·시간순 분할 결과를 보관한다.
for group in GROUPS:
    df_group = prep_builder.load(group, "train")
    target_col = f"kpx_group_{group}"
    feature_cols = WindforceDatasetBuilder.featureCols(df_group)
    X_train, X_valid, y_train, y_valid = splitByTime(
        df_group,
        target_col,
        feature_cols,
    )
    datasets[group] = {
        "df": df_group,
        "feature_cols": feature_cols,
        "X_train": X_train,
        "X_valid": X_valid,
        "y_train": y_train,
        "y_valid": y_valid,
    }
    # 모든 모델이 같은 행·피처·검증 구간을 재사용하도록 한곳에 고정한다.
    print(
        f"[그룹{group}] 전체 {df_group.shape} | 피처 {len(feature_cols)}개 | "
        f"train {len(X_train):,} / valid {len(X_valid):,}"
    )
    # 그룹별 데이터 shape와 공통 분할 크기를 확인한다.


---
## Step 2. LightGBM 전체/SHAP선별/FICR목적함수 학습 + 예측 저장소 (카드 A·C)

먼저 전체 피처로 LightGBM(`LGBM_full`)을 학습한다. 검증 표본의 평균 절대 SHAP 기여도와
무작위 노이즈 기준선을 계산해 노이즈 이하 피처·절대 상관 0.95 초과 중복 피처를 제거하고,
같은 하이퍼파라미터로 선별 피처 모델(`LGBM_selected`)을 다시 학습한다.

**카드 C(FICR-aware 목적함수)**: `LGBM_selected`와 같은 선별 피처로 세 번째 후보
`LGBM_ficr`를 학습한다. `regression_l1`은 FICR 신호를 전혀 받지 않으므로, 1단계 L1 웜업
→ 2단계 FICR 혼합 목적함수(`grad`/`hess`를 직접 유도)로 이어 학습해 LSTM의
`ScoreLossFunction` 웜업 설계와 원칙을 맞춘다(`baseline7_idea_evaluation.md` 4-1~4-3절).

**카드 A(예측 저장소)**: 이 Step에서 만드는 모든 후보의 검증 구간 예측을
`predictions_store[(group, model_name)] = (pred, actual)`에 저장해, 이후 카드 B/D/E/F가
재학습 없이 재사용한다.

모든 모델 지표는 LSTM과 같은 시각을 쓰기 위해 검증 구간의 첫 `SEQ_LEN=24`시간을 제외한다.


In [ ]:
LGB_PARAMS = {
    "objective": "regression_l1",
    # NMAE가 절대오차 기반이므로 학습 목적을 L1로 맞춘다.
    "n_estimators": 1_200,
    # 실제 트리 수는 검증 L1 조기종료가 결정한다.
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 40,
    # 파워커브 비선형을 표현하되 작은 노이즈 구간의 과적합을 억제한다.
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    # 행·열 샘플링과 L2 정규화로 트리 간 상관을 낮춘다.
    "random_state": SEED,
    "n_jobs": -1,
    "verbose": -1,
}
# 전체/선별/FICR 후보 모두 같은 설정을 적용해 피처 선택·목적함수 효과만 비교한다.

metrics = EvaluationMetrics(
    rated_capacity_kw=RATED_CAPACITY_KW,
    time_step_hours=TIME_STEP_HOURS,
)
# KPX 그룹별 설비용량(kW)과 1시간 해상도를 쓰는 공식 지표 계산기


def trainLightGBM(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_valid: pd.DataFrame,
    y_valid: np.ndarray,
    capacity_kw: float,
) -> lgb.LGBMRegressor:
    """설비용량 대비 이용률로 LightGBM을 학습하는 함수"""
    model = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(
        X_train,
        y_train / capacity_kw,
        # 발전량(kWh)을 설비용량(kW) 대비 0~1 이용률로 정규화한다.
        eval_X=X_valid,
        eval_y=y_valid / capacity_kw,
        eval_metric="l1",
        callbacks=[
            lgb.early_stopping(80, verbose=False),
            lgb.log_evaluation(0),
        ],
        # 검증 L1이 80개 트리 동안 개선되지 않으면 최적 반복에서 멈춘다.
    )
    return model
    # best_iteration_이 기록된 학습 모델을 반환한다.


def predictKw(
    model: lgb.LGBMRegressor,
    X: pd.DataFrame,
    capacity_kw: float,
) -> np.ndarray:
    """LightGBM 이용률 예측을 발전량(kWh)으로 환산하는 함수"""
    utilization = np.clip(model.predict(X), 0.0, 1.0)
    # 물리적으로 가능한 설비 이용률 0~1 범위로 제한한다.
    return utilization * capacity_kw
    # 1시간 자료이므로 이용률 × 설비용량(kW) = 시간당 발전량(kWh)이다.


def summarizeAll(
    pred: np.ndarray,
    actual: np.ndarray,
    group: int,
) -> dict:
    """공식 지표와 MAE·RMSE(kWh)를 함께 반환하는 함수"""
    summary = metrics.summarize(pred, actual, group)
    # NMAE·FICR·그룹 총점을 대회 정의 그대로 계산한다.
    summary["mae"] = float(np.mean(np.abs(pred - actual)))
    summary["rmse"] = float(np.sqrt(np.mean((pred - actual) ** 2)))
    # 직관적인 오차 규모를 시간당 발전량(kWh) 단위로 추가한다.
    return summary
    # 리더보드 입력 스키마로 합칠 지표 딕셔너리를 반환한다.


In [ ]:
class FeatureSelector:
    """null importance와 상관 중복으로 SHAP 피처를 선별하는 클래스

    사용 순서:
        selector = FeatureSelector(capacity_kw=RATED_CAPACITY_KW[1])
        columns = selector.select(X_train, y_train, X_valid, y_valid, shap_importance)
    """

    def __init__(
        self,
        capacity_kw: float,
        corr_threshold: float = CORR_THRESHOLD,
        noise_count: int = NOISE_FEATURE_COUNT,
        cumulative_ratio: float = CUMULATIVE_RATIO,
        seed: int = SEED,
    ):
        self.capacity_kw = capacity_kw
        # 이용률 학습에 쓰는 그룹 설비용량(kW)
        self.corr_threshold = corr_threshold
        # 절대 상관 중복 판정 임계값
        self.noise_count = noise_count
        # null importance 기준선용 무작위 피처 개수
        self.cumulative_ratio = cumulative_ratio
        # 상위 SHAP 누적 기여 유지 비율
        self.seed = seed
        # 무작위 노이즈 재현 시드
        self.noise_threshold_: Optional[float] = None
        self.report_: Optional[pd.DataFrame] = None
        # select() 실행 뒤 채워지는 노이즈 기준과 피처별 판정표

    @staticmethod
    def _shapArray(model: lgb.LGBMRegressor, sample: pd.DataFrame) -> np.ndarray:
        """SHAP 버전별 반환 형식을 2차원 배열로 통일하는 내부 메서드"""
        values = shap.TreeExplainer(model).shap_values(sample)
        # 트리 구조를 직접 순회해 표본 × 피처 SHAP 기여도를 계산한다.
        if isinstance(values, list):
            values = values[0]
        # 일부 SHAP 버전이 회귀에서도 list를 반환하는 경우 첫 출력을 사용한다.
        return np.asarray(values)
        # shape = (표본 수, 피처 수)인 numpy 배열로 반환한다.

    def _computeNoiseImportance(
        self,
        X_train: pd.DataFrame,
        y_train: np.ndarray,
        X_valid: pd.DataFrame,
        y_valid: np.ndarray,
        sample_index: pd.Index,
    ) -> pd.Series:
        """노이즈 피처를 포함한 모델의 평균 절대 SHAP 기여도를 계산하는 내부 메서드"""
        rng = np.random.default_rng(self.seed)
        X_train_noise = X_train.copy()
        X_valid_noise = X_valid.copy()
        # 호출자가 보관한 공통 분할 DataFrame을 변형하지 않는다.
        for index in range(self.noise_count):
            X_train_noise[f"__noise_{index}"] = rng.normal(size=len(X_train_noise))
            X_valid_noise[f"__noise_{index}"] = rng.normal(size=len(X_valid_noise))
            # 타깃과 무관한 표준정규 피처를 학습·검증 양쪽에 독립 생성한다.

        model = trainLightGBM(
            X_train_noise,
            y_train,
            X_valid_noise,
            y_valid,
            self.capacity_kw,
        )
        sample = X_valid_noise.loc[sample_index]
        values = self._shapArray(model, sample)
        importance = pd.Series(
            np.abs(values).mean(axis=0),
            index=X_train_noise.columns,
        )
        # 방향이 상쇄되지 않도록 표본 평균 |SHAP|을 피처 중요도로 사용한다.
        noise_cols = [f"__noise_{index}" for index in range(self.noise_count)]
        self.noise_threshold_ = float(importance[noise_cols].max())
        # 노이즈 중 최대 기여도를 우연히 얻을 수 있는 보수적 하한으로 정한다.
        return importance
        # 원래 피처와 노이즈 피처의 같은 척도 중요도를 반환한다.

    def select(
        self,
        X_train: pd.DataFrame,
        y_train: np.ndarray,
        X_valid: pd.DataFrame,
        y_valid: np.ndarray,
        shap_importance: pd.Series,
        sample_index: pd.Index,
    ) -> list[str]:
        """노이즈·상관·누적 기여 기준으로 피처를 선별하는 메서드"""
        feature_cols = X_train.columns.tolist()
        # 원래 학습 피처 순서를 판정표 작성에 보존한다.
        noise_importance = self._computeNoiseImportance(
            X_train,
            y_train,
            X_valid,
            y_valid,
            sample_index,
        )
        survived = [
            column
            for column in feature_cols
            if noise_importance.get(column, 0.0) > self.noise_threshold_
        ]
        dropped_noise = set(feature_cols) - set(survived)
        # 노이즈 최대 기여도 이하인 피처를 정보 없는 후보로 제거한다.

        corr = X_train[survived].corr().abs()
        # 검증 통계를 보지 않고 학습 구간의 절대 상관만 계산한다.
        ordered = [column for column in shap_importance.index if column in survived]
        selected: list[str] = []
        dropped_corr: dict[str, str] = {}
        for column in ordered:
            duplicate = next(
                (
                    kept
                    for kept in selected
                    if corr.loc[column, kept] > self.corr_threshold
                ),
                None,
            )
            if duplicate is None:
                selected.append(column)
            else:
                dropped_corr[column] = duplicate
            # SHAP 상위 피처를 먼저 남기고 같은 정보를 가진 하위 피처를 제거한다.

        ranked = shap_importance.loc[selected]
        if self.cumulative_ratio < 1.0:
            cumulative = ranked.cumsum() / ranked.sum()
            keep_count = int((cumulative < self.cumulative_ratio).sum()) + 1
            selected = ranked.index[:keep_count].tolist()
        # 1.0이면 baseline 5 HOWTO 실측대로 꼬리 절단을 적용하지 않는다.

        rows = []
        for column in feature_cols:
            if column in dropped_noise:
                decision = "노이즈 이하"
            elif column in dropped_corr:
                decision = f"중복({dropped_corr[column]})"
            elif column in selected:
                decision = "선택"
            else:
                decision = "누적 기여 꼬리"
            rows.append(
                {
                    "feature": column,
                    "shap_importance": float(shap_importance.get(column, 0.0)),
                    "decision": decision,
                }
            )
            # 피처별 제거·선택 사유를 재현 가능한 표로 기록한다.
        self.report_ = pd.DataFrame(rows).sort_values(
            "shap_importance",
            ascending=False,
        ).reset_index(drop=True)
        # 사람이 상위 피처와 제거 사유를 함께 검토할 수 있게 정렬한다.
        return selected
        # 평균 |SHAP| 내림차순의 최종 피처 목록을 반환한다.


In [ ]:
def makeFicrObjective(k: float = 40.0, regression_weight: float = 0.7):
    """LightGBM용 grad(오차 방향)와 hess(오차 변화 민감도)를 반환하는 목적함수 생성 함수

    Args:
        - k: softFICR 시그모이드 steepness(경사). ScoreLossFunction과 같은 기본값(40.0) 사용
        - regression_weight: L1 항 가중치. 1이면 순수 L1(기존과 동일), 0이면 순수 FICR 근사

    Logic:
        - 타깃·예측 모두 설비이용률(0~1) 단위이므로 오차율 e = pred - actual을 그대로 쓴다
        - FICR(오차가 작을수록 발전량에 더 높은 정산단가를 적용하는 점수)의 계단 문턱(6%/8%)을
          시그모이드 2개로 완화한 뒤 미분해 grad/hess를 유도한다
          (baseline7_idea_evaluation.md 4-1절에서 손으로 재검증됨)
        - hess 하한을 절대값(1e-6)이 아니라 regression_weight에 비례한 상대 하한으로 잡는다.
          문턱 근처 소수 표본에서 hess가 수백 배 커지는 비균질 분포가 되면 LightGBM
          리프 가중치(-Σgrad/(Σhess+λ))가 그 소수 표본에 지배되어 발산할 수 있기 때문이다
          (baseline7_idea_evaluation.md 4-2절 근거)
    """
    def objective(y_true: np.ndarray, y_pred: np.ndarray):
        e = y_pred - y_true
        s = np.sign(e)
        r = np.abs(e)

        g1 = 1.0 / (1.0 + np.exp(-k * (r - 0.06)))
        g2 = 1.0 / (1.0 + np.exp(-k * (r - 0.08)))

        d_ficr = s * (k * g1 * (1 - g1) + 3.0 * k * g2 * (1 - g2)) / 4.0
        dd_ficr = (
            k * k * g1 * (1 - g1) * (1 - 2 * g1)
            + 3.0 * k * k * g2 * (1 - g2) * (1 - 2 * g2)
        ) / 4.0

        grad = regression_weight * s + (1.0 - regression_weight) * d_ficr
        hess_floor = max(1e-2 * (1.0 - regression_weight), 1e-6)
        # 상대 하한: regression_weight가 클수록(=FICR 항 비중이 작을수록) 하한도 낮춘다
        hess = np.maximum((1.0 - regression_weight) * np.abs(dd_ficr), hess_floor)
        return grad, hess
    return objective


def trainFicrLightGBM(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_valid: pd.DataFrame,
    y_valid: np.ndarray,
    capacity_kw: float,
    warmup_rounds: int = 200,
    k: float = 40.0,
    regression_weight: float = 0.7,
) -> lgb.LGBMRegressor:
    """1단계 L1 웜업 → 2단계 FICR 혼합 목적함수로 이어 학습하는 함수

    Args:
        - warmup_rounds: 1단계(순수 L1) 부스팅 라운드 수
        - k, regression_weight: makeFicrObjective에 그대로 전달할 하이퍼파라미터

    Logic:
        - 1단계(warmup_rounds): objective="regression_l1"로 대략적인 예측 수준을 먼저 잡는다
          (LSTM의 SmoothL1 워밍업(warmup_epochs)과 같은 목적 — 학습 초반 그래디언트 안정화)
        - 2단계: init_model=1단계 booster로 이어받아 makeFicrObjective로 계속 부스팅한다
        - min_sum_hessian_in_leaf를 기본값(1e-3)보다 높여(0.05) hess 총합이 작은 리프의
          추가 분할을 억제한다(baseline7_idea_evaluation.md 4-2절 완화책)
    """
    warmup_params = {**LGB_PARAMS, "objective": "regression_l1", "n_estimators": warmup_rounds}
    model_warmup = lgb.LGBMRegressor(**warmup_params)
    model_warmup.fit(
        X_train, y_train / capacity_kw,
        eval_X=X_valid, eval_y=y_valid / capacity_kw,
        eval_metric="l1", callbacks=[lgb.log_evaluation(0)],
    )
    # 1단계는 조기종료 없이 warmup_rounds만큼 고정 학습해 2단계의 안정된 출발점을 만든다.

    ficr_params = {
        **LGB_PARAMS,
        "objective": makeFicrObjective(k, regression_weight),
        "min_sum_hessian_in_leaf": 0.05,
    }
    model_ficr = lgb.LGBMRegressor(**ficr_params)
    model_ficr.fit(
        X_train, y_train / capacity_kw,
        eval_X=X_valid, eval_y=y_valid / capacity_kw,
        eval_metric="l1", init_model=model_warmup.booster_,
        callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)],
    )
    return model_ficr
    # 웜업 부스터를 이어받아 FICR 혼합 목적함수로 계속 학습한 최종 모델을 반환한다.


In [ ]:
records: list[dict] = []
# 그룹·모델별 공통 지표 행을 누적한다.
lgb_models: dict[tuple[int, str], lgb.LGBMRegressor] = {}
selected_features: dict[int, list[str]] = {}
selection_reports: dict[int, pd.DataFrame] = {}
shap_store: dict[int, tuple[np.ndarray, pd.DataFrame, pd.Series]] = {}
# 최종 학습과 해석에 재사용할 모델·피처·SHAP 산출물을 보관한다.
predictions_store: dict[tuple[int, str], tuple[np.ndarray, np.ndarray]] = {}
# (그룹, 모델명) -> (검증 구간 예측 kWh, 실제 kWh). 카드 B~F가 재학습 없이 이 저장소만 읽어서
# 동작한다(baseline7_howto_next.md 카드 A). 평가(2025년) 구간 예측은 여기 섞지 않는다.

for group in GROUPS:
    data = datasets[group]
    X_train = data["X_train"]
    X_valid = data["X_valid"]
    y_train = data["y_train"]
    y_valid = data["y_valid"]
    capacity_kw = RATED_CAPACITY_KW[group]
    actual_aligned = y_valid[SEQ_LEN:]
    # 모든 모델을 LSTM과 같은 검증 시각으로 비교하기 위해 앞 24시간을 제외한다.

    model_full = trainLightGBM(
        X_train,
        y_train,
        X_valid,
        y_valid,
        capacity_kw,
    )
    lgb_models[(group, "LGBM_full")] = model_full
    pred_full = predictKw(model_full, X_valid, capacity_kw)[SEQ_LEN:]
    summary_full = summarizeAll(pred_full, actual_aligned, group)
    predictions_store[(group, "LGBM_full")] = (pred_full, actual_aligned)
    records.append(
        {
            "group": group,
            "model_name": "LGBM_full",
            "n_features": X_train.shape[1],
            "best_epoch": model_full.best_iteration_,
            **summary_full,
        }
    )
    # 전체 피처 LightGBM을 기준 모델과 최종 후보로 보관한다.

    sample = X_valid.sample(
        min(SHAP_SAMPLE_SIZE, len(X_valid)),
        random_state=SEED,
    )
    shap_values = FeatureSelector._shapArray(model_full, sample)
    shap_importance = pd.Series(
        np.abs(shap_values).mean(axis=0),
        index=X_train.columns,
    ).sort_values(ascending=False)
    shap_store[group] = (shap_values, sample, shap_importance)
    # 검증 표본의 평균 절대 SHAP 기여도를 내림차순으로 기록한다.

    selector = FeatureSelector(capacity_kw=capacity_kw)
    columns = selector.select(
        X_train,
        y_train,
        X_valid,
        y_valid,
        shap_importance,
        sample.index,
    )
    selected_features[group] = columns
    selection_reports[group] = selector.report_
    # 그룹마다 신호 분포가 다르므로 피처 선별도 별도로 수행한다.

    model_selected = trainLightGBM(
        X_train[columns],
        y_train,
        X_valid[columns],
        y_valid,
        capacity_kw,
    )
    lgb_models[(group, "LGBM_selected")] = model_selected
    pred_selected = predictKw(
        model_selected,
        X_valid[columns],
        capacity_kw,
    )[SEQ_LEN:]
    summary_selected = summarizeAll(pred_selected, actual_aligned, group)
    predictions_store[(group, "LGBM_selected")] = (pred_selected, actual_aligned)
    records.append(
        {
            "group": group,
            "model_name": "LGBM_selected",
            "n_features": len(columns),
            "best_epoch": model_selected.best_iteration_,
            **summary_selected,
        }
    )
    # 같은 설정으로 선별 피처 모델을 다시 학습해 선택 효과를 분리한다.

    model_ficr = trainFicrLightGBM(
        X_train[columns], y_train, X_valid[columns], y_valid,
        capacity_kw=capacity_kw, k=40.0, regression_weight=0.7,
    )
    lgb_models[(group, "LGBM_ficr")] = model_ficr
    pred_ficr = predictKw(model_ficr, X_valid[columns], capacity_kw)[SEQ_LEN:]
    summary_ficr = summarizeAll(pred_ficr, actual_aligned, group)
    predictions_store[(group, "LGBM_ficr")] = (pred_ficr, actual_aligned)
    records.append({
        "group": group, "model_name": "LGBM_ficr", "n_features": len(columns),
        "best_epoch": model_ficr.best_iteration_, **summary_ficr,
    })
    # 카드 C: 같은 선별 피처에 FICR-aware 목적함수만 바꿔 학습한 세 번째 후보

    persistence = BaselineModels.persistenceForecast(y_valid)[SEQ_LEN:]
    summary_persistence = summarizeAll(persistence, actual_aligned, group)
    predictions_store[(group, "Persistence_oracle_lag1")] = (persistence, actual_aligned)
    records.append(
        {
            "group": group,
            "model_name": "Persistence_oracle_lag1",
            "n_features": 0,
            "best_epoch": np.nan,
            **summary_persistence,
        }
    )
    # 직전 실제 발전량을 쓰는 oracle은 비교선으로만 기록하고 제출 선택에서는 제외한다.

    print(
        f"[그룹{group}] 전체 {X_train.shape[1]} → 선별 {len(columns)}개 | "
        f"LGBM_full={summary_full['score']:.4f} | "
        f"LGBM_selected={summary_selected['score']:.4f} | "
        f"LGBM_ficr={summary_ficr['score']:.4f}"
    )
    # 그룹별 피처 감소와 공식 총점 변화를 한 줄로 확인한다.

selection_path = f"{PREP_DIR}/baseline8_selected_features.csv"
selection_df = pd.concat(
    [
        pd.DataFrame(
            {
                "group": group,
                "feature": selected_features[group],
                "rank": range(1, len(selected_features[group]) + 1),
            }
        )
        for group in GROUPS
    ],
    ignore_index=True,
)
selection_df.to_csv(selection_path, index=False, encoding="utf-8-sig")
# 그룹별 SHAP 선별 피처와 순위를 LSTM 단계 전부터 재사용할 수 있게 저장한다.
print(f"✅ 피처 저장: {selection_path}")
# 긴 LSTM 실험을 나중에 실행해도 선별 결과를 먼저 보존한다.


In [ ]:
# 그룹별 상위 SHAP 피처와 선택 사유를 확인한다.
for group in GROUPS:
    print(f"=== KPX 그룹 {group}: SHAP 상위 15개 ===")
    display(selection_reports[group].head(15))
    # 중요도와 선택/제거 사유를 같은 표에서 읽는다.

    shap_values, sample, _ = shap_store[group]
    shap.summary_plot(shap_values, sample, max_display=15, show=False)
    plt.title(f"KPX 그룹 {group} — SHAP Summary 상위 15개")
    plt.xlabel("SHAP value (설비 이용률 기여도)")
    plt.tight_layout()
    plt.show()
    # 점의 색은 피처값, 가로 위치는 이용률 예측에 대한 기여 방향을 나타낸다.


---
## Step 3. 동일 선별 피처로 LSTM v1/v2/v3 공정 비교 + 예측 저장소 (카드 A)

baseline 6의 세 LSTM 표현 전략만 남기고, 입력은 Step 2의 그룹별 SHAP 선별 피처로 통일한다.

| 버전 | 표현 방식 |
|---|---|
| `v1` | 마지막 시각 hidden state |
| `v2` | LayerNorm + Dropout |
| `v3` | 24개 시각 Attention 가중합 |

결측 대체와 MinMax 스케일링은 학습 구간에만 적합한다. 조기종료 기준은 MAE가 아니라
그룹별 공식 `0.5×(1-NMAE)+0.5×FICR`로 맞춘다. 각 버전의 검증 예측도
`predictions_store`(카드 A)에 저장해 카드 B/D/E/F가 재사용한다.


In [ ]:
class OfficialScoreMonitor:
    """VersionedLSTMPipeline의 검증 monitor를 공식 그룹 총점으로 연결하는 클래스"""

    def __init__(self, metrics: EvaluationMetrics, group: int):
        self.metrics = metrics
        # 계단형 FICR을 그대로 계산하는 공식 지표 객체
        self.group = group
        # 이 LSTM이 담당하는 KPX 그룹 번호

    def score(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        capacity_kw: Optional[float] = None,
    ) -> float:
        """실제값·예측값으로 그룹 공식 총점을 반환하는 메서드"""
        del capacity_kw
        # 파이프라인 인터페이스 호환용 인자이며 그룹 용량은 metrics 내부 상수를 사용한다.
        return float(
            self.metrics.summarize(y_pred, y_true, self.group)["score"]
        )
        # 공식 그룹 총점이 클수록 좋은 조기종료 monitor를 반환한다.


MODEL_VERSIONS = ["v1", "v2", "v3"]
# baseline 6의 세 시퀀스 표현 전략
MODEL_PARAMS = {
    "epochs": 80,
    "warmup_epochs": 15,
    "loss_k": 40.0,
    "regression_weight": 0.75,
    "patience": 12,
    "batch_size": 256,
    # 64 → 256으로 에폭당 스텝을 1/4로 줄인다.
    "lr": 2e-3,
    # 배치가 4배가 되면 학습률을 sqrt(4)=2배로 올려 수렴 속도를 보정한다.
    "random_state": SEED,
}
# baseline 7의 안정화 설정으로 통일해 구조 이외의 차이를 줄인다.

lstm_artifacts: dict[tuple[int, str], dict] = {}
# 그룹·버전별 결측 변환기·스케일러·학습 파이프라인을 보관한다.

for group in GROUPS:
    data = datasets[group]
    columns = selected_features[group]
    X_train = data["X_train"][columns]
    X_valid = data["X_valid"][columns]
    y_train = data["y_train"]
    y_valid = data["y_valid"]
    # LightGBM과 동일한 SHAP 선별 피처·시간순 분할을 사용한다.

    variant = AllFeaturesVariant(random_state=SEED)
    train_frame = variant.fit_transform(X_train, y_train)
    valid_frame = variant.transform(X_valid)
    # 학습 중앙값으로 결측을 채우고 검증 컬럼 순서를 학습 계약에 맞춘다.
    scaler = MinMaxScaler().fit(train_frame)
    X_train_scaled = scaler.transform(train_frame)
    X_valid_scaled = scaler.transform(valid_frame)
    # MinMaxScaler는 학습 구간에만 fit해 검증 통계 누수를 차단한다.

    monitor = OfficialScoreMonitor(metrics, group)
    # 이 그룹의 NMAE·FICR 공식 총점을 조기종료 기준으로 사용한다.
    for version in MODEL_VERSIONS:
        pipeline = make_lstm_pipeline(
            version,
            capacity_kw=RATED_CAPACITY_KW[group],
            seq_len=SEQ_LEN,
            **MODEL_PARAMS,
        )
        pipeline.fit(
            X_train_scaled,
            y_train,
            X_val=X_valid_scaled,
            y_val=y_valid,
            metrics=monitor,
            group=group,
        )
        # 같은 입력·학습값에서 LSTM 표현 전략만 바꿔 공식 총점으로 조기종료한다.
        pred = pipeline.predict(X_valid_scaled, y_valid)
        actual = y_valid[SEQ_LEN:]
        # 슬라이딩 윈도우가 소비한 첫 24시간을 실제값에서도 동일하게 제외한다.
        summary = summarizeAll(pred, actual, group)
        model_name = f"LSTM_{version}_shap_selected"
        predictions_store[(group, model_name)] = (pred, actual)
        records.append(
            {
                "group": group,
                "model_name": model_name,
                "n_features": X_train_scaled.shape[1],
                "best_epoch": pipeline.best_epoch,
                **summary,
            }
        )
        lstm_artifacts[(group, model_name)] = {
            "variant": variant,
            "scaler": scaler,
            "pipeline": pipeline,
        }
        # 최종 후보 판단과 best_epoch 재사용을 위해 학습 객체를 보관한다.
        print(
            f"[그룹{group}][{version}] epoch={pipeline.best_epoch} | "
            f"NMAE={summary['nmae']:.4f} FICR={summary['ficr']:.4f} "
            f"총점={summary['score']:.4f}"
        )
        # 그룹·버전별 공식 지표를 공통 형식으로 출력한다.


---
## Step 4. 공식 총점 집계 + 그룹별 최적 모델 선택 + 오차 밴드 진단 (카드 A)

결과 CSV는 baseline 7과 호환되는 핵심 컬럼을 유지하고 `best_epoch`를 추가한다.
최종 제출 후보는 Persistence를 제외하고 그룹별 검증 총점이 가장 높은 모델을 선택한다.

**카드 A(오차율 밴드 진단)**: `predictions_store`의 모든 (그룹, 모델) 조합에 대해 시간별
오차율을 FICR 정산단가 구간(`≤6%`/`6~8%`/`>8%`)으로 나눠 비율을 계산하고, 그 결과로 카드
B(사후 보정)와 카드 C(재학습) 중 어느 쪽에 자원을 먼저 투입할지 규칙 기반으로 추천한다.


In [ ]:
def aggregateOfficialScore(result_df: pd.DataFrame) -> pd.DataFrame:
    """모델별 3그룹 평균 NMAE·FICR로 공식 총점을 집계하는 함수"""
    summary = result_df.groupby("model_name", as_index=False).agg(
        mean_nmae=("nmae", "mean"),
        mean_ficr=("ficr", "mean"),
        mean_mae=("mae", "mean"),
        mean_rmse=("rmse", "mean"),
        mean_features=("n_features", "mean"),
        groups=("group", "nunique"),
    )
    # 모델별로 세 그룹 지표와 피처 수를 먼저 평균한다.
    summary["official_score"] = (
        0.5 * (1.0 - summary["mean_nmae"]) + 0.5 * summary["mean_ficr"]
    )
    # 대회 정의에 따라 평균 NMAE·FICR을 같은 비중으로 합친다.
    return summary.sort_values("official_score", ascending=False).reset_index(drop=True)
    # 제출 후보가 위로 오도록 공식 총점 내림차순으로 반환한다.


result_df = pd.DataFrame(records).drop_duplicates(
    subset=["group", "model_name"],
    keep="last",
)
# 셀을 다시 실행해도 같은 그룹·모델 행이 중복 누적되지 않게 마지막 기록만 남긴다.
summary_df = aggregateOfficialScore(result_df)
# 세 그룹을 모두 고려한 모델 계열별 공식 총점표를 만든다.

selectable = result_df[
    result_df["model_name"] != "Persistence_oracle_lag1"
].copy()
# 미래 실제값이 필요한 oracle 모델은 제출 후보에서 제외한다.
best_by_group = (
    selectable.sort_values(["group", "score"], ascending=[True, False])
    .groupby("group", as_index=False)
    .first()
)
# 각 그룹의 검증 총점 최고 제출 가능 모델을 한 행씩 선택한다.

display(summary_df)
print("=== 그룹별 최종 제출 모델(현재까지 후보 중) ===")
display(best_by_group[["group", "model_name", "n_features", "score"]])
# 전체 모델 순위와 실제 제출에 쓸 그룹별 후보를 함께 확인한다.

results_path = f"{ROOT}/BASELINE/baseline_8_results.csv"
summary_path = f"{ROOT}/BASELINE/baseline_8_summary.csv"
result_df.to_csv(results_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
# 그룹 원본 지표와 세 그룹 공식 집계를 리더보드 재사용 형식으로 저장한다.
print(f"✅ 결과 저장: {results_path}")
print(f"✅ 요약 저장: {summary_path}")
# 생성된 두 산출물 경로를 명시한다.

for group in GROUPS:
    gap = best_by_group.loc[best_by_group["group"] == group, "score"].iloc[0] - BASELINE7_SCORE[group]
    print(f"[그룹{group}] baseline_7 기준선({BASELINE7_SCORE[group]:.4f}) 대비 {gap:+.4f}")
    # 카드 B~F 착수 전, LGBM_ficr까지 포함한 현재 최고 후보가 baseline_7을 넘는지 먼저 확인한다.


In [ ]:
# 모델별 세 그룹 공식 총점을 같은 축에서 비교한다.
plot_df = summary_df.sort_values("official_score")
colors = [
    "#B0B0B0" if "Persistence" in name else "#2E7D9A"
    for name in plot_df["model_name"]
]
# 제출 불가능한 Persistence만 회색, 나머지 후보는 파란색으로 표시한다.
plt.figure(figsize=(10, 0.55 * len(plot_df) + 2.0))
bars = plt.barh(plot_df["model_name"], plot_df["official_score"], color=colors)
for bar, value in zip(bars, plot_df["official_score"]):
    plt.text(
        value + 0.004,
        bar.get_y() + bar.get_height() / 2.0,
        f"{value:.4f}",
        va="center",
    )
    # 각 막대 오른쪽에 공식 총점 소수점 4자리를 표시한다.
plt.xlabel("Official Total Score 0.5×(1-NMAE) + 0.5×FICR (bigger is better)")
plt.title("Baseline 8 — LightGBM·LSTM Total validation Score by Model")
plt.xlim(0.0, max(plot_df["official_score"]) * 1.15)
plt.tight_layout()
plt.show()
# 모델 계열별 3그룹 평균 성능을 시각적으로 비교한다.


In [ ]:
def summarizeErrorBand(pred: np.ndarray, actual: np.ndarray, group: int) -> pd.Series:
    """시간별 오차율을 FICR(오차가 작을수록 단가가 높은 점수) 구간으로 나눠 비율을 계산하는 함수

    Args:
        - pred, actual: 검증 구간 예측·실제 발전량(kWh) 배열
        - group: KPX 그룹 번호

    Returns:
        - 인덱스 ["≤6%","6~8%",">8%"], 값은 각 구간에 속하는 시간 비율(합 1.0)인 Series

    Logic:
        - hourlyRatio = |pred - actual| / RATED_CAPACITY_KW[group]
        - EvaluationMetrics.evaluateFICR과 동일한 문턱(0.06/0.08)으로 구간을 나눈다
        - ">8%" 비율이 클수록 그 시간대는 정산단가 0원 → FICR 손실의 직접 원인
    """
    ratio = np.abs(pred - actual) / RATED_CAPACITY_KW[group]
    band = pd.Series(pd.cut(
        ratio,
        bins=[-np.inf, 0.06, 0.08, np.inf],
        labels=["≤6%", "6~8%", ">8%"],
    ))
    # pd.cut(ndarray)는 Series가 아닌 Categorical을 반환하므로, Series.value_counts(normalize=...)를
    # 쓰기 위해 명시적으로 pd.Series로 감싼다(Categorical.value_counts는 normalize 인자가 없다).
    return band.value_counts(normalize=True).reindex(["≤6%", "6~8%", ">8%"])
    # 세 구간의 시간 비율 합이 1이 되는 Series를 반환한다


def recommendAction(band_share: pd.Series, gt8_dominance_ratio: float = 1.5,
                     gt8_absolute_floor: float = 0.15, mid_band_floor: float = 0.10) -> str:
    """오차 밴드 비율로부터 카드 B/C 중 무엇을 우선할지 규칙 기반으로 추천하는 함수

    Args:
        - band_share: summarizeErrorBand()의 반환값, 인덱스 ["≤6%","6~8%",">8%"]
        - gt8_dominance_ratio: ">8%"가 "6~8%"의 이 배수 이상이면 카드 C(재학습) 우선
        - gt8_absolute_floor: ">8%" 절대 비율이 이 값 이상이면 배수와 무관하게 카드 C 우선
        - mid_band_floor: ">8%"는 낮지만 "6~8%"가 이 값 이상이면 카드 B(보정)만으로 개선 여지

    Logic:
        - ">8%"는 FICR 기여가 0원이므로 이 비율을 줄이는 것이 총점에 가장 직접적이다.
          그래서 ">8%" 조건을 "6~8%" 조건보다 먼저 검사한다(우선순위 비대칭).
        - 세 조건 모두 해당 없으면 이미 양호하므로 원안(best_by_group) 유지를 권고한다.
        - 이 함수의 출력은 "권고"일 뿐 강제가 아니다. 카드 B는 비용이 거의 없으므로 권고와
          무관하게 항상 먼저 시도하고, 이 함수는 "카드 C에 자원을 더 투자할 가치가 있는가"만
          판단하는 데 쓴다(baseline7_idea_evaluation.md 아이디어 3-3 근거).
    """
    gt8, mid = band_share[">8%"], band_share["6~8%"]
    if gt8 >= gt8_absolute_floor or (mid > 0 and gt8 >= gt8_dominance_ratio * mid):
        return "카드 C 우선 (근본적 재학습 — FICR-aware 목적함수)"
    if mid >= mid_band_floor:
        return "카드 B 우선 (저비용 사후 보정으로 6~8%를 6% 이하로 밀어넣기 시도)"
    return "카드 B/C 착수 전 원안 유지 — 이미 밴드 분포 양호"


band_rows = []
for (group, model_name), (pred, actual) in predictions_store.items():
    share = summarizeErrorBand(pred, actual, group)
    band_rows.append({
        "group": group, "model_name": model_name, **share.to_dict(),
        "recommended_action": recommendAction(share),
    })
band_df = pd.DataFrame(band_rows).sort_values(["group", ">8%"])
display(band_df)
# 그룹·모델별 밴드 분포와 권고 조치를 한 표로 비교한다

print("=== best_by_group 후보의 밴드 진단 ===")
for _, row in best_by_group.iterrows():
    match = band_df[(band_df["group"] == row["group"]) & (band_df["model_name"] == row["model_name"])]
    if not match.empty:
        print(
            f"[그룹{row['group']}] {row['model_name']} | "
            f">8%={match['>8%'].iloc[0]:.3f} | {match['recommended_action'].iloc[0]}"
        )
    # 카드 B~F 착수 전, 각 그룹이 보정 우선인지 재학습 우선인지 실행 로그로 남긴다.


---
## Step 5. 사후 보정(post-hoc calibration) — 전역 (카드 B)

재학습 비용 없이 예측값에 단조 아핀(기울기·절편) 보정만 적용해 검증 총점을 높일 수 있는지
확인한다. 비용이 거의 없으므로 카드 C(재학습)보다 먼저 시도하는 것이 원칙이지만, 이 노트북은
카드 C(`LGBM_ficr`)를 이미 Step 2에서 함께 학습해 두었으므로, 여기서는 `predictions_store`의
모든 후보(원본 + `LGBM_ficr`)에 전역 보정을 적용해 비교한다. 시각대별 확장은 Step 7(카드 E)의
불균일 진단 이후에 별도로 시도한다.


In [ ]:
class FicrCalibrator:
    """예측값에 단조 아핀(기울기와 절편으로 만드는 직선) 보정을 적용하는 클래스

    사용 순서:
        calibrator = FicrCalibrator(capacity_kw=RATED_CAPACITY_KW[group]).fit(pred, actual, group)
        pred_calibrated = calibrator.transform(pred)

    Logic:
        - pred_calibrated = clip(a * pred + b, 0, capacity_kw)
        - (a, b) 격자탐색으로 검증 구간 공식 총점이 최대인 조합을 찾는다
        - 반드시 fit()에 넘긴 예측·실제값(검증 구간)에서만 (a, b)를 찾는다. 평가(2025년)
          구간에서는 다시 fit()하지 않고 여기서 고정한 (a, b)를 transform()에만 쓴다
    """
    A_GRID = np.linspace(0.85, 1.15, 21)
    B_RATIO_GRID = np.linspace(-0.02, 0.02, 21)
    # b는 설비용량 비율로 탐색한 뒤 capacity_kw를 곱해 kWh 단위로 환산한다

    def __init__(self, capacity_kw: float):
        self.capacity_kw = capacity_kw
        self.a_: float = 1.0
        self.b_: float = 0.0

    def fit(self, pred: np.ndarray, actual: np.ndarray, group: int) -> "FicrCalibrator":
        best_score, best_a, best_b = -np.inf, 1.0, 0.0
        for a in self.A_GRID:
            for b_ratio in self.B_RATIO_GRID:
                b = b_ratio * self.capacity_kw
                candidate = np.clip(a * pred + b, 0.0, self.capacity_kw)
                score = metrics.summarize(candidate, actual, group)["score"]
                if score > best_score:
                    best_score, best_a, best_b = score, a, b
        # 441개 조합은 그룹당 수 초 내에 끝나므로 벡터화 없이 이중 루프로 충분하다
        self.a_, self.b_ = float(best_a), float(best_b)
        return self

    def transform(self, pred: np.ndarray) -> np.ndarray:
        return np.clip(self.a_ * pred + self.b_, 0.0, self.capacity_kw)


class DiurnalFicrCalibrator(FicrCalibrator):
    """target_hour_ldaps 3구간(단기/중기/장기)별로 (a, b)를 독립적으로 탐색하는 확장 클래스

    Logic:
        - 구간마다 표본 수가 카드 A 대비 1/3로 줄어드므로, 표본이 너무 적으면(<500시간)
          과적합 위험이 있다는 것을 fit() 호출부에서 로그로 알린다(카드 E 검증 가설 참고)
        - 구간별 (a_h, b_h)를 dict로 보관하고, transform()은 hour_bucket 배열을 받아
          구간별로 다른 변환을 적용한다
    """
    MIN_BUCKET_HOURS = 500

    def fit(self, pred: np.ndarray, actual: np.ndarray, group: int,
            hour_bucket: np.ndarray) -> "DiurnalFicrCalibrator":
        self.params_: dict[str, tuple[float, float]] = {}
        for bucket in np.unique(hour_bucket):
            mask = hour_bucket == bucket
            if mask.sum() < self.MIN_BUCKET_HOURS:
                print(f"⚠️ 그룹{group} {bucket} 표본이 {int(mask.sum())}시간뿐 — 과적합 위험")
            sub = FicrCalibrator(self.capacity_kw).fit(pred[mask], actual[mask], group)
            self.params_[bucket] = (sub.a_, sub.b_)
        return self

    def transform(self, pred: np.ndarray, hour_bucket: np.ndarray) -> np.ndarray:
        out = pred.copy()
        for bucket, (a, b) in self.params_.items():
            mask = hour_bucket == bucket
            out[mask] = np.clip(a * pred[mask] + b, 0.0, self.capacity_kw)
        return out


In [ ]:
calibration_rows = []
calibrators: dict[tuple[int, str], FicrCalibrator] = {}
# 카드 F 최종 채택 시 재사용할 (그룹, 모델명) -> 적합된 보정기

for (group, model_name), (pred, actual) in predictions_store.items():
    if model_name == "Persistence_oracle_lag1":
        continue
    # 미래 실제값을 쓰는 oracle은 제출 후보가 아니므로 보정 대상에서 제외한다.
    before = metrics.summarize(pred, actual, group)
    calibrator = FicrCalibrator(capacity_kw=RATED_CAPACITY_KW[group]).fit(pred, actual, group)
    after = metrics.summarize(calibrator.transform(pred), actual, group)
    calibrators[(group, model_name)] = calibrator
    calibration_rows.append({
        "group": group, "model_name": model_name,
        "score_before": before["score"], "score_after": after["score"],
        "ficr_before": before["ficr"], "ficr_after": after["ficr"],
        "a": calibrator.a_, "b_kw": calibrator.b_,
    })
calibration_df = pd.DataFrame(calibration_rows).sort_values("score_after", ascending=False)
display(calibration_df)
# score_after < score_before인 행은 채택하지 않는다(항등 변환 a=1,b=0이 이미 최선이라는 뜻).

print("=== best_by_group 후보의 전역 보정 효과 ===")
for _, row in best_by_group.iterrows():
    match = calibration_df[
        (calibration_df["group"] == row["group"])
        & (calibration_df["model_name"] == row["model_name"])
    ]
    if not match.empty:
        gain = match["score_after"].iloc[0] - match["score_before"].iloc[0]
        print(
            f"[그룹{row['group']}] {row['model_name']} | "
            f"score_before={match['score_before'].iloc[0]:.4f} → "
            f"score_after={match['score_after'].iloc[0]:.4f} ({gain:+.4f})"
        )
    # 카드 F에서 이 그룹을 보정 채택할지 판단할 근거를 실행 로그로 남긴다.


---
## Step 6. 지능형 하이퍼파라미터 최적화 — GWO (카드 D)

기존 3×3 그리드 탐색은 두 파라미터의 실제 최적점이 격자점 사이에 있으면 놓친다. `.venv`에
`optuna`/`scikit-optimize` 같은 베이지안 최적화 라이브러리가 없으므로, **GWO(Grey Wolf
Optimizer, 여러 후보 설정을 늑대 무리처럼 움직이며 탐색하는 방법)를 순수 numpy로 구현**해
연속 공간을 탐색한다(`baseline7_idea_evaluation.md` 아이디어 6).

탐색 대상은 (1) LSTM `loss_k`/`regression_weight`(그룹별 이미 정한 버전 1개만 재탐색)와
(2) LightGBM `LGBM_ficr`의 `k`/`regression_weight` 두 가지다. 두 탐색 모두 검증 구간
(`X_valid`/`y_valid`) 총점만 목적함수로 쓰고, 평가(2025년) 구간은 어떤 형태로도 노출하지
않는다. GWO 결과가 기존 후보보다 좋으면 `LSTM_{version}_gwo` / `LGBM_ficr_gwo`라는 이름으로
`predictions_store`·`records`에 새 후보로 등록해 카드 F가 다른 후보와 동일하게 비교하게 한다.


In [ ]:
class GreyWolfOptimizer:
    """GWO(후보 설정을 무리처럼 비교하는 방법)로 하이퍼파라미터를 탐색하는 클래스

    사용 순서:
        gwo = GreyWolfOptimizer(bounds=[(10,100),(0.5,0.95)], n_wolves=5, max_iter=4, seed=SEED)
        best_params, best_score, history = gwo.optimize(objective_fn)

    Logic:
        - 개체(늑대) 위치 = 하이퍼파라미터 벡터. alpha/beta/delta(상위 3개체)의 위치를
          기준으로 나머지 개체가 이동하는 군집 지능 알고리즘
        - a는 매 반복 2에서 0으로 선형 감소 (탐색→수렴 전환을 제어하는 유일한 스케줄 파라미터)
        - objective_fn은 "클수록 좋음"(검증 총점)을 반환한다고 가정하고 내부에서 최대화한다
        - 표준 GWO는 엘리트 보존이 없어 반복 중 best_score가 감소할 수 있으므로, optimize()는
          매 반복의 최댓값이 아니라 history 전체에서 최댓값을 최종 채택한다(안전장치)
    """
    def __init__(self, bounds: list[tuple[float, float]], n_wolves: int = 5,
                 max_iter: int = 4, seed: int = SEED):
        self.bounds = np.asarray(bounds, dtype=np.float64)
        self.n_wolves, self.max_iter = n_wolves, max_iter
        self.rng = np.random.default_rng(seed)

    def _clip(self, pos: np.ndarray) -> np.ndarray:
        return np.clip(pos, self.bounds[:, 0], self.bounds[:, 1])

    def optimize(self, objective_fn):
        dim = len(self.bounds)
        lo, hi = self.bounds[:, 0], self.bounds[:, 1]
        positions = lo + self.rng.random((self.n_wolves, dim)) * (hi - lo)
        # 탐색 범위 내 균등 무작위 초기화

        scores = np.array([objective_fn(p) for p in positions])
        history = [{"iter": 0, "best_score": float(scores.max()), "best_pos": positions[scores.argmax()].copy()}]

        for it in range(self.max_iter):
            order = np.argsort(-scores)
            # 검증 총점 내림차순: 상위 3개체(alpha/beta/delta)가 나머지를 이끈다
            alpha, beta, delta = positions[order[0]], positions[order[1]], positions[order[2]]
            a = 2.0 - 2.0 * it / max(1, self.max_iter - 1)
            # a: 반복이 진행될수록 2->0으로 선형 감소, 탐색(exploration)에서 수렴(exploitation)으로 전환

            for i in range(self.n_wolves):
                new_pos = np.zeros(dim)
                for leader in (alpha, beta, delta):
                    r1, r2 = self.rng.random(dim), self.rng.random(dim)
                    A = 2 * a * r1 - a
                    C = 2 * r2
                    D = np.abs(C * leader - positions[i])
                    new_pos += leader - A * D
                positions[i] = self._clip(new_pos / 3.0)
                # alpha/beta/delta 각각이 이끄는 위치의 평균으로 이동 (표준 GWO 갱신식)

            scores = np.array([objective_fn(p) for p in positions])
            history.append({"iter": it + 1, "best_score": float(scores.max()), "best_pos": positions[scores.argmax()].copy()})

        best_record = max(history, key=lambda row: row["best_score"])
        # 엘리트 보존이 없는 표준 GWO는 마지막 반복이 최고라는 보장이 없으므로 history 전체에서 고른다
        return best_record["best_pos"], float(best_record["best_score"]), history


In [ ]:
gwo_started = time.time()
grid_targets = {1: "v1", 2: "v2", 3: "v1"}
# baseline 6 원안과 동일한 그룹별 재탐색 대상 버전(그룹1: v1, 그룹2: v2, 그룹3: v1)
gwo_rows = []
gwo_best_pipelines: dict[int, dict] = {}
# 그룹별 GWO 최적 파이프라인·스케일러·variant를 카드 F 채택 판단에 재사용하기 위해 보관한다

for group, version in grid_targets.items():
    data = datasets[group]
    columns = selected_features[group]
    X_train, X_valid = data["X_train"][columns], data["X_valid"][columns]
    y_train, y_valid = data["y_train"], data["y_valid"]
    variant = AllFeaturesVariant(random_state=SEED)
    train_frame = variant.fit_transform(X_train, y_train)
    valid_frame = variant.transform(X_valid)
    scaler = MinMaxScaler().fit(train_frame)
    X_train_scaled, X_valid_scaled = scaler.transform(train_frame), scaler.transform(valid_frame)
    monitor = OfficialScoreMonitor(metrics, group)

    def objective_fn(params, group=group, version=version):
        loss_k, regression_weight = float(params[0]), float(params[1])
        pipeline = make_lstm_pipeline(
            version, capacity_kw=RATED_CAPACITY_KW[group], seq_len=SEQ_LEN,
            **{**MODEL_PARAMS, "loss_k": loss_k, "regression_weight": regression_weight},
        )
        pipeline.fit(X_train_scaled, y_train, X_val=X_valid_scaled, y_val=y_valid,
                     metrics=monitor, group=group)
        pred = pipeline.predict(X_valid_scaled, y_valid)
        actual = y_valid[SEQ_LEN:]
        return metrics.summarize(pred, actual, group)["score"]

    gwo = GreyWolfOptimizer(bounds=[(10.0, 100.0), (0.5, 0.95)], n_wolves=5, max_iter=4, seed=SEED)
    best_params, best_score, history = gwo.optimize(objective_fn)
    gwo_rows.append({
        "group": group, "version": version,
        "loss_k": best_params[0], "regression_weight": best_params[1],
        "best_score": best_score, "n_evals": len(history) * 5,
        "monotone_history": all(
            history[i]["best_score"] <= history[i + 1]["best_score"] + 1e-9
            for i in range(len(history) - 1)
        ),
    })
    # n_evals: 개체 수 x (반복 수 + 초기화 1회)만큼 objective_fn이 호출된 횟수 — 실행시간 추정용
    gwo_best_pipelines[group] = {
        "version": version, "variant": variant, "scaler": scaler,
        "loss_k": float(best_params[0]), "regression_weight": float(best_params[1]),
    }
    print(f"[그룹{group}][{version}] GWO best_score={best_score:.4f} "
          f"(k={best_params[0]:.1f}, w={best_params[1]:.3f}) | 누적 {time.time() - gwo_started:.1f}초")

gwo_df = pd.DataFrame(gwo_rows)
display(gwo_df)
print(f"LSTM GWO 총 소요: {time.time() - gwo_started:.1f}초")
# §4 리스크 관리: 3그룹 예상 20~27분 기준. 크게 초과하면 다음 실행부터 n_wolves=4, max_iter=3으로 축소한다.


In [ ]:
# GWO가 그리드 탐색(원안 MODEL_PARAMS 고정값)보다 나쁜 지역해에 수렴하지 않았는지 확인하고,
# 개선된 그룹만 재학습한 LSTM을 predictions_store/records에 새 후보로 등록한다.
for group, version in grid_targets.items():
    row = gwo_df.loc[gwo_df["group"] == group].iloc[0]
    baseline_candidate_score = best_by_group.loc[best_by_group["group"] == group, "score"].iloc[0]
    if row["best_score"] <= baseline_candidate_score:
        print(f"[그룹{group}] GWO 개선 없음({row['best_score']:.4f} <= {baseline_candidate_score:.4f}) — 채택 보류")
        continue
    # 검증 총점이 기존 최고 후보를 넘을 때만 재학습·등록한다(불필요한 추가 학습 방지).

    info = gwo_best_pipelines[group]
    data = datasets[group]
    columns = selected_features[group]
    X_train_scaled = info["scaler"].transform(info["variant"].transform(data["X_train"][columns]))
    X_valid_scaled = info["scaler"].transform(info["variant"].transform(data["X_valid"][columns]))
    y_train, y_valid = data["y_train"], data["y_valid"]
    monitor = OfficialScoreMonitor(metrics, group)

    pipeline = make_lstm_pipeline(
        info["version"], capacity_kw=RATED_CAPACITY_KW[group], seq_len=SEQ_LEN,
        **{**MODEL_PARAMS, "loss_k": info["loss_k"], "regression_weight": info["regression_weight"]},
    )
    pipeline.fit(X_train_scaled, y_train, X_val=X_valid_scaled, y_val=y_valid, metrics=monitor, group=group)
    pred = pipeline.predict(X_valid_scaled, y_valid)
    actual = y_valid[SEQ_LEN:]
    summary = summarizeAll(pred, actual, group)

    model_name = f"LSTM_{info['version']}_gwo"
    predictions_store[(group, model_name)] = (pred, actual)
    records.append({
        "group": group, "model_name": model_name, "n_features": X_train_scaled.shape[1],
        "best_epoch": pipeline.best_epoch, **summary,
    })
    lstm_artifacts[(group, model_name)] = {
        "variant": info["variant"], "scaler": info["scaler"], "pipeline": pipeline,
        "loss_k": info["loss_k"], "regression_weight": info["regression_weight"],
    }
    print(f"✅ [그룹{group}] {model_name} 등록 | 총점={summary['score']:.4f}")


In [ ]:
# LightGBM LGBM_ficr의 k/regression_weight도 동일한 GWO 루프로 탐색한다(LightGBM은 LSTM보다
# 학습이 훨씬 빠르므로 n_wolves=8, max_iter=6으로 예산을 더 크게 잡는다).
lgbm_gwo_started = time.time()
lgbm_gwo_rows = []
lgbm_gwo_best_params: dict[int, dict] = {}

for group in GROUPS:
    data = datasets[group]
    columns = selected_features[group]
    X_train, X_valid = data["X_train"][columns], data["X_valid"][columns]
    y_train, y_valid = data["y_train"], data["y_valid"]
    capacity_kw = RATED_CAPACITY_KW[group]
    actual_aligned = y_valid[SEQ_LEN:]

    def objective_fn(params, capacity_kw=capacity_kw, actual_aligned=actual_aligned):
        k, regression_weight = float(params[0]), float(params[1])
        model = trainFicrLightGBM(
            X_train, y_train, X_valid, y_valid,
            capacity_kw=capacity_kw, k=k, regression_weight=regression_weight,
        )
        pred = predictKw(model, X_valid, capacity_kw)[SEQ_LEN:]
        return metrics.summarize(pred, actual_aligned, group)["score"]

    gwo = GreyWolfOptimizer(bounds=[(10.0, 100.0), (0.5, 0.95)], n_wolves=8, max_iter=6, seed=SEED)
    best_params, best_score, history = gwo.optimize(objective_fn)
    lgbm_gwo_rows.append({
        "group": group, "k": best_params[0], "regression_weight": best_params[1],
        "best_score": best_score, "n_evals": len(history) * 8,
    })
    lgbm_gwo_best_params[group] = {"k": float(best_params[0]), "regression_weight": float(best_params[1])}
    print(f"[그룹{group}] LGBM GWO best_score={best_score:.4f} "
          f"(k={best_params[0]:.1f}, w={best_params[1]:.3f}) | 누적 {time.time() - lgbm_gwo_started:.1f}초")

lgbm_gwo_df = pd.DataFrame(lgbm_gwo_rows)
display(lgbm_gwo_df)
print(f"LGBM GWO 총 소요: {time.time() - lgbm_gwo_started:.1f}초")


In [ ]:
# LGBM_ficr GWO 결과가 원안(k=40.0, w=0.7)보다 좋은 그룹만 재학습해 LGBM_ficr_gwo로 등록한다.
for group in GROUPS:
    row = lgbm_gwo_df.loc[lgbm_gwo_df["group"] == group].iloc[0]
    baseline_ficr_score = result_df.loc[
        (result_df["group"] == group) & (result_df["model_name"] == "LGBM_ficr"), "score"
    ].iloc[0]
    if row["best_score"] <= baseline_ficr_score:
        print(f"[그룹{group}] LGBM GWO 개선 없음({row['best_score']:.4f} <= {baseline_ficr_score:.4f}) — 채택 보류")
        continue

    data = datasets[group]
    columns = selected_features[group]
    X_train, X_valid = data["X_train"][columns], data["X_valid"][columns]
    y_train, y_valid = data["y_train"], data["y_valid"]
    capacity_kw = RATED_CAPACITY_KW[group]
    actual_aligned = y_valid[SEQ_LEN:]
    params = lgbm_gwo_best_params[group]

    model = trainFicrLightGBM(
        X_train, y_train, X_valid, y_valid,
        capacity_kw=capacity_kw, k=params["k"], regression_weight=params["regression_weight"],
    )
    pred = predictKw(model, X_valid, capacity_kw)[SEQ_LEN:]
    summary = summarizeAll(pred, actual_aligned, group)

    model_name = "LGBM_ficr_gwo"
    lgb_models[(group, model_name)] = model
    predictions_store[(group, model_name)] = (pred, actual_aligned)
    records.append({
        "group": group, "model_name": model_name, "n_features": len(columns),
        "best_epoch": model.best_iteration_, **summary,
    })
    print(f"✅ [그룹{group}] {model_name} 등록 | 총점={summary['score']:.4f}")


---
## Step 7. 시각대(diurnal) 오차 진단 매트릭스 (카드 E) + 사후 보정 시각대 확장 (카드 B)

`target_hour_ldaps`(예보 대상 시각, 1~24시)는 이 데이터셋에서 시각(hour-of-day)의 결정론적
아핀 변환과 동일한 정보이고(`baseline7_idea_evaluation.md` 2-1절, 상관계수 1.0 실측), 이미
SHAP 선별을 통과해 두 모델 계열 모두의 학습 입력에 들어가 있다. 따라서 이 카드의 실제 작업은
"모델에 새 정보를 주는 것"이 아니라 **시간별 오차율의 밴드 분포가 시각대별로 실제로 불균일한지
진단**하고, 불균일할 때만 카드 B를 시각대 차원(`DiurnalFicrCalibrator`)으로 확장하는 것이다.

불균일 판정: 어떤 시각대의 `>8%` 비율이 다른 시각대 평균보다 **1.5배 이상** 크면 "불균일"로
판정한다. 불균일이 아닌 그룹은 시각대별 보정을 시도하지 않고 전역 보정(Step 5)만 유지한다
(`baseline7_idea_evaluation.md` 2-4절 ROI 표 — 데이터 손실 대비 효용이 낮다는 결론을 따른다).


In [ ]:
# GWO(카드 D)가 추가한 후보까지 포함해 그룹별 현재 최고 후보를 다시 뽑는다.
result_df = pd.DataFrame(records).drop_duplicates(subset=["group", "model_name"], keep="last")
summary_df = aggregateOfficialScore(result_df)
selectable = result_df[result_df["model_name"] != "Persistence_oracle_lag1"].copy()
best_by_group = (
    selectable.sort_values(["group", "score"], ascending=[True, False])
    .groupby("group", as_index=False)
    .first()
)
print("=== 카드 D 반영 후 그룹별 최고 후보 ===")
display(best_by_group[["group", "model_name", "n_features", "score"]])


In [ ]:
HOUR_BINS = [0, 8, 16, 24]
HOUR_LABELS = ["단기(01~08시)", "중기(09~16시)", "장기(17~24시)"]
# target_hour_ldaps(1~24시)를 3구간으로 나눈다. "리드타임"이 아니라 "시각대" 축이라는 점에
# 유의한다(baseline7_idea_evaluation.md 2-1절 — 이 데이터셋에서 리드타임은 시각의 아핀
# 변환과 상관계수 1.0으로 동일한 정보다).

diurnal_rows = []
diurnal_imbalance: dict[int, float] = {}
diurnal_hour_bucket: dict[int, np.ndarray] = {}
# 카드 B 시각대 확장이 재사용할 그룹별 hour_bucket(ndarray)을 보관한다

for group in GROUPS:
    model_name = best_by_group.loc[best_by_group["group"] == group, "model_name"].iloc[0]
    pred, actual = predictions_store[(group, model_name)]
    hour = datasets[group]["X_valid"]["target_hour_ldaps"].to_numpy()[SEQ_LEN:]
    # Step 2·3 모두 SEQ_LEN만큼 앞을 잘라 검증했으므로 시각 배열도 동일하게 정렬한다.
    hour_bucket = pd.cut(hour, bins=HOUR_BINS, labels=HOUR_LABELS, include_lowest=True).to_numpy()
    # pd.cut(ndarray)의 반환은 Categorical이며 .to_numpy()로 plain ndarray로 바꿔두면
    # 이후 (hour_bucket == bucket) 비교와 DiurnalFicrCalibrator 전달 모두 안전하다.
    diurnal_hour_bucket[group] = hour_bucket

    gt8_shares = []
    for bucket in HOUR_LABELS:
        mask = hour_bucket == bucket
        if mask.sum() == 0:
            continue
        share = summarizeErrorBand(pred[mask], actual[mask], group)
        gt8_shares.append(share[">8%"])
        diurnal_rows.append({
            "group": group, "model_name": model_name, "hour_bucket": bucket,
            "n_hours": int(mask.sum()), **share.to_dict(),
        })
    avg_gt8 = float(np.mean(gt8_shares)) if gt8_shares else 0.0
    diurnal_imbalance[group] = (max(gt8_shares) / avg_gt8) if avg_gt8 > 0 else 1.0
    # 어떤 시각대의 ">8%" 비율이 평균 대비 몇 배인지로 불균일 정도를 잰다.

diurnal_df = pd.DataFrame(diurnal_rows)
display(diurnal_df)

DIURNAL_IMBALANCE_THRESHOLD = 1.5
diurnal_uneven_groups = [
    group for group, ratio in diurnal_imbalance.items() if ratio >= DIURNAL_IMBALANCE_THRESHOLD
]
print(f"시각대별 >8% 비율 불균형(최대/평균): {diurnal_imbalance}")
print(
    f"→ 불균일 판정(≥{DIURNAL_IMBALANCE_THRESHOLD}배) 그룹: {diurnal_uneven_groups or '없음'} "
    "— 이 그룹에서만 시각대별 사후 보정을 시도한다."
)


In [ ]:
# 카드 B 확장: 불균일 그룹만 DiurnalFicrCalibrator를 시도하고, 전역 보정 대비 더 나을 때만
# calibrators 딕셔너리를 시각대별 보정기로 교체한다.
diurnal_adopted_pairs: set[tuple[int, str]] = set()
# 이 집합에 든 (그룹, 모델명)만 calibrators[...]가 DiurnalFicrCalibrator다 — 카드 F가 transform()
# 호출 시 (pred,) / (pred, hour_bucket) 중 어떤 시그니처를 써야 하는지 이 집합으로 구분한다.
diurnal_calibration_rows = []
for group in diurnal_uneven_groups:
    model_name = best_by_group.loc[best_by_group["group"] == group, "model_name"].iloc[0]
    pred, actual = predictions_store[(group, model_name)]
    hour_bucket = diurnal_hour_bucket[group]

    global_after = metrics.summarize(calibrators[(group, model_name)].transform(pred), actual, group)["score"]
    # Step 5에서 이미 적합해 둔 전역 보정기의 검증 총점(비교 기준)

    diurnal_calibrator = DiurnalFicrCalibrator(capacity_kw=RATED_CAPACITY_KW[group]).fit(
        pred, actual, group, hour_bucket,
    )
    diurnal_after = metrics.summarize(diurnal_calibrator.transform(pred, hour_bucket), actual, group)["score"]

    diurnal_calibration_rows.append({
        "group": group, "model_name": model_name,
        "score_global": global_after, "score_diurnal": diurnal_after,
        "adopt_diurnal": diurnal_after > global_after,
    })
    if diurnal_after > global_after:
        calibrators[(group, model_name)] = diurnal_calibrator
        diurnal_adopted_pairs.add((group, model_name))
        # 시각대별 보정이 전역 보정보다 나을 때만 카드 F가 참조할 보정기를 교체하고 표시한다.

diurnal_calibration_df = pd.DataFrame(diurnal_calibration_rows)
if len(diurnal_calibration_df) > 0:
    display(diurnal_calibration_df)
else:
    print("불균일 그룹이 없어 시각대별 보정을 시도하지 않았습니다 — 전역 보정(Step 5)을 유지합니다.")


---
## Step 8. 최종 후보 채택 + 앙상블 (카드 F)

카드 A~E에서 나온 모든 후보 — 원본(`LGBM_full`/`LGBM_selected`/`LSTM_*`), 카드 C
(`LGBM_ficr`), 카드 D(`*_gwo`) — 를 그룹별로 다시 모아 전역/시각대별 사후 보정을 새로
적합하고(카드 B), 검증 총점 내림차순으로 정렬한다. 후보가 2개 이상 기준선을 넘으면 가중
블렌딩(앙상블)도 시도한다. **채택 기준은 항상 baseline_7 실측 그룹별 기준선**이며, 넘지
못하면 그 그룹은 원안(`best_by_group`)을 그대로 유지한다.


In [ ]:
def bestPredFor(group: int, model_name: str) -> tuple[np.ndarray, np.ndarray]:
    """후보의 원본·보정 검증 예측 중 총점이 더 높은 쪽을 돌려주는 헬퍼 함수

    Returns:
        - (pred, actual): candidate_df가 이미 판정한 더 나은 쪽의 검증 구간 예측·실제값
    """
    pred, actual = predictions_store[(group, model_name)]
    row = candidate_df[
        (candidate_df["group"] == group) & (candidate_df["model_name"] == model_name)
    ].iloc[0]
    if row["score_calibrated"] <= row["score_raw"]:
        return pred, actual
    calibrator = final_calibrators[(group, model_name)]
    if final_calibrator_kind[(group, model_name)] == "diurnal":
        return calibrator.transform(pred, diurnal_hour_bucket[group]), actual
    return calibrator.transform(pred), actual


final_result_df = pd.DataFrame(records).drop_duplicates(subset=["group", "model_name"], keep="last")
final_result_df = final_result_df[final_result_df["model_name"] != "Persistence_oracle_lag1"]
# 미래 실제값이 필요한 oracle은 카드 F 최종 채택 대상에서 제외한다.

candidate_rows = []
final_calibrators: dict[tuple[int, str], object] = {}
final_calibrator_kind: dict[tuple[int, str], str] = {}
# (그룹, 모델명) -> 새로 적합한 보정기 / "global" 또는 "diurnal"(불균일 그룹만)

for group in GROUPS:
    group_rows = final_result_df[final_result_df["group"] == group]
    for _, row in group_rows.iterrows():
        model_name = row["model_name"]
        pred, actual = predictions_store[(group, model_name)]
        raw_score = float(row["score"])

        calibrator = FicrCalibrator(capacity_kw=RATED_CAPACITY_KW[group]).fit(pred, actual, group)
        calibrated_score = metrics.summarize(calibrator.transform(pred), actual, group)["score"]
        kind = "global"

        if group in diurnal_uneven_groups:
            hour_bucket = diurnal_hour_bucket[group]
            d_calibrator = DiurnalFicrCalibrator(capacity_kw=RATED_CAPACITY_KW[group]).fit(
                pred, actual, group, hour_bucket,
            )
            d_score = metrics.summarize(d_calibrator.transform(pred, hour_bucket), actual, group)["score"]
            if d_score > calibrated_score:
                calibrator, calibrated_score, kind = d_calibrator, d_score, "diurnal"
            # 불균일 그룹만 시각대별 보정을 시도하고, 전역 보정보다 나을 때만 교체한다.

        final_calibrators[(group, model_name)] = calibrator
        final_calibrator_kind[(group, model_name)] = kind
        candidate_rows.append({
            "group": group, "model_name": model_name,
            "score_raw": raw_score, "score_calibrated": float(calibrated_score),
            "calibration_kind": kind,
            "best_score": max(raw_score, float(calibrated_score)),
        })

candidate_df = pd.DataFrame(candidate_rows).sort_values(
    ["group", "best_score"], ascending=[True, False],
).reset_index(drop=True)
display(candidate_df)
# 그룹별 모든 후보를 원본/보정 최고 점수 기준 내림차순으로 나열한다.


In [ ]:
# 후보가 2개 이상 baseline_7 기준선을 넘은 그룹에서만 가중 블렌딩(앙상블)을 시도한다.
ensemble_rows = []
for group in GROUPS:
    group_candidates = candidate_df[candidate_df["group"] == group].reset_index(drop=True)
    beats_baseline = group_candidates[group_candidates["best_score"] > BASELINE7_SCORE[group]]
    if len(beats_baseline) < 2:
        continue
    # 후보가 하나뿐이면 블렌딩할 상대가 없으므로 앙상블을 시도하지 않는다.

    a_name, b_name = beats_baseline.iloc[0]["model_name"], beats_baseline.iloc[1]["model_name"]
    pred_a, actual = bestPredFor(group, a_name)
    pred_b, _ = bestPredFor(group, b_name)
    a_row = group_candidates[group_candidates["model_name"] == a_name].iloc[0]
    b_row = group_candidates[group_candidates["model_name"] == b_name].iloc[0]
    a_calibrated = bool(a_row["score_calibrated"] > a_row["score_raw"])
    b_calibrated = bool(b_row["score_calibrated"] > b_row["score_raw"])
    # 평가 구간에서도 같은 보정 적용 여부를 재현할 수 있도록 각 구성 후보의 보정 채택 여부를 남긴다.

    best_w, best_score = 0.5, -np.inf
    for w in np.arange(0.0, 1.01, 0.05):
        blended = w * pred_a + (1 - w) * pred_b
        score = metrics.summarize(blended, actual, group)["score"]
        if score > best_score:
            best_score, best_w = float(score), float(w)
        # 가중치는 검증 예측에서만 탐색한다(평가 구간은 여기서 전혀 쓰지 않는다).

    solo_best = float(beats_baseline.iloc[0]["best_score"])
    ensemble_rows.append({
        "group": group, "a": a_name, "b": b_name, "w": best_w,
        "ensemble_score": best_score, "solo_best": solo_best,
        "a_calibrated": bool(a_calibrated), "b_calibrated": bool(b_calibrated),
        "adopt": best_score > solo_best,
    })
    # 앙상블 총점이 두 후보 각각의 단독 최고 점수보다 낮으면 채택하지 않는다(§4 리스크 관리).

ensemble_df = pd.DataFrame(ensemble_rows)
if len(ensemble_df) > 0:
    display(ensemble_df)
else:
    print("2개 이상 기준선을 넘은 그룹이 없어 앙상블을 시도하지 않았습니다.")


In [ ]:
final_choice: dict[int, dict] = {}
# 그룹별 최종 채택 정보. kind in {"raw","calibrated","ensemble"}

for group in GROUPS:
    group_candidates = candidate_df[candidate_df["group"] == group].reset_index(drop=True)
    top_row = group_candidates.iloc[0]
    chosen = {
        "kind": "calibrated" if top_row["score_calibrated"] > top_row["score_raw"] else "raw",
        "model_name": top_row["model_name"],
        "score": float(top_row["best_score"]),
    }

    if len(ensemble_df) > 0:
        match = ensemble_df[(ensemble_df["group"] == group) & (ensemble_df["adopt"])]
        if not match.empty and match.iloc[0]["ensemble_score"] > chosen["score"]:
            row = match.iloc[0]
            chosen = {
                "kind": "ensemble",
                "model_name": f"{row['a']}+{row['b']}(w={row['w']:.2f})",
                "score": float(row["ensemble_score"]),
                "ensemble_a": row["a"], "ensemble_b": row["b"], "ensemble_w": float(row["w"]),
                "a_calibrated": bool(row["a_calibrated"]), "b_calibrated": bool(row["b_calibrated"]),
            }
        # 앙상블이 채택 조건(두 후보보다 높음)을 만족하고, 단독 최고 후보보다도 높을 때만 교체한다.

    if chosen["score"] <= BASELINE7_SCORE[group]:
        original_name = best_by_group.loc[best_by_group["group"] == group, "model_name"].iloc[0]
        original_score = float(best_by_group.loc[best_by_group["group"] == group, "score"].iloc[0])
        chosen = {"kind": "raw", "model_name": original_name, "score": original_score}
        print(
            f"[그룹{group}] 어떤 후보도 기준선({BASELINE7_SCORE[group]:.4f})을 넘지 못해 "
            f"원안({original_name}, {original_score:.4f}) 유지"
        )
        # baseline_7 실측 기준선을 넘지 못하면 카드 B~F를 적용하지 않은 원안을 그대로 쓴다.
    else:
        print(f"[그룹{group}] 최종 채택: {chosen['kind']} {chosen['model_name']} (총점 {chosen['score']:.4f})")
    final_choice[group] = chosen

display(pd.DataFrame([{"group": g, **v} for g, v in final_choice.items()]))


---
## Step 9. 전체 학습 + 평가 기간 예측 + 제출 검증 (카드 F 반영)

Step 8(`final_choice`)에서 그룹별로 정한 채택 방식(원본 / 사후보정 / 앙상블)을 그대로 반영해
2022~2024년 전체 학습 데이터로 다시 적합하고 2025년 예측을 만든다. 사후 보정·앙상블 가중치는
**검증 구간에서 이미 고정한 값을 그대로 적용만 하고 평가 구간에서 다시 탐색하지 않는다.**
결측을 `0`이나 직전값으로 숨기지 않고 예측 길이·시각 정렬이 어긋나면 즉시 중단한다.


In [ ]:
def fitFinalLightGBM(
    X_full: pd.DataFrame,
    y_full: np.ndarray,
    capacity_kw: float,
    n_estimators: int,
) -> lgb.LGBMRegressor:
    """검증에서 정한 트리 수로 전체 학습 데이터에 순수 L1 LightGBM을 적합하는 함수"""
    params = {**LGB_PARAMS, "n_estimators": max(1, int(n_estimators))}
    # 검증 best_iteration_을 전체 학습의 고정 트리 수로 사용한다.
    model = lgb.LGBMRegressor(**params)
    model.fit(X_full, y_full / capacity_kw)
    # 전체 기간의 발전량(kWh)을 그룹 설비 이용률로 정규화해 학습한다.
    return model
    # 평가 기간 예측에 사용할 최종 트리 모델을 반환한다.


def fitFinalFicrLightGBM(
    X_full: pd.DataFrame,
    y_full: np.ndarray,
    capacity_kw: float,
    warmup_rounds: int,
    n_estimators: int,
    k: float,
    regression_weight: float,
) -> lgb.LGBMRegressor:
    """검증에서 정한 라운드 수로 전체 학습 데이터에 2단계 FICR LightGBM(카드 C)을 적합하는 함수

    Logic:
        - trainFicrLightGBM과 같은 1단계(L1 웜업) → 2단계(FICR 혼합 목적함수) 구조를 쓰되,
          검증 구간이 더 이상 없으므로 조기종료 없이 검증에서 정한 고정 라운드 수로 학습한다
        - fitFinalLightGBM과 같은 "검증에서 트리 수를 정하고 전체 데이터로 재학습" 원칙을 따른다
    """
    warmup_params = {**LGB_PARAMS, "objective": "regression_l1", "n_estimators": warmup_rounds}
    model_warmup = lgb.LGBMRegressor(**warmup_params)
    model_warmup.fit(X_full, y_full / capacity_kw)

    ficr_params = {
        **LGB_PARAMS,
        "objective": makeFicrObjective(k, regression_weight),
        "n_estimators": max(1, int(n_estimators)),
        "min_sum_hessian_in_leaf": 0.05,
    }
    model_ficr = lgb.LGBMRegressor(**ficr_params)
    model_ficr.fit(X_full, y_full / capacity_kw, init_model=model_warmup.booster_)
    return model_ficr
    # 웜업 부스터를 이어받아 FICR 혼합 목적함수로 계속 학습한 최종 모델을 반환한다.


def fitFinalCandidate(group: int, model_name: str, train_df: pd.DataFrame, test_df: pd.DataFrame) -> np.ndarray:
    """검증에서 채택된 임의의 후보를 전체 데이터로 재학습해 평가 구간 예측(kWh)을 반환하는 함수

    Args:
        - group: KPX 그룹 번호
        - model_name: predictions_store/records에 쓰인 후보 이름
          (`LGBM_full`/`LGBM_selected`/`LGBM_ficr`/`LGBM_ficr_gwo`/`LSTM_v{1,2,3}_shap_selected`/
          `LSTM_v{1,2,3}_gwo` 중 하나)
        - train_df: 이 그룹의 2022~2024년 전체 학습 표
        - test_df: 이 그룹의 2025년 평가 표 (8,760행)

    Returns:
        - np.ndarray, shape=(8_760,) — 평가 구간 예측 발전량(kWh)

    Raises:
        - KeyError: 평가 데이터에 학습에 쓴 피처가 없는 경우
    """
    target_col = f"kpx_group_{group}"
    y_full = train_df[target_col].to_numpy(dtype=float)
    capacity_kw = RATED_CAPACITY_KW[group]

    if model_name.startswith("LGBM"):
        columns = datasets[group]["feature_cols"] if model_name == "LGBM_full" else selected_features[group]
        missing = [column for column in columns if column not in test_df.columns]
        if missing:
            raise KeyError(f"[그룹{group}] 평가 데이터에 없는 피처: {missing}")
        # 평가 컬럼 불일치를 조용히 재색인하지 않고 즉시 알린다.

        best_iteration = lgb_models[(group, model_name)].best_iteration_
        if model_name in ("LGBM_ficr", "LGBM_ficr_gwo"):
            if model_name == "LGBM_ficr":
                k, regression_weight = 40.0, 0.7
            else:
                params = lgbm_gwo_best_params[group]
                k, regression_weight = params["k"], params["regression_weight"]
            final_model = fitFinalFicrLightGBM(
                train_df[columns], y_full, capacity_kw,
                warmup_rounds=200, n_estimators=best_iteration,
                k=k, regression_weight=regression_weight,
            )
        else:
            final_model = fitFinalLightGBM(train_df[columns], y_full, capacity_kw, best_iteration)
        return predictKw(final_model, test_df[columns], capacity_kw)
        # 검증에서 정한 트리 수·목적함수로 전체 기간을 학습하고 8,760시간을 예측한다.

    columns = selected_features[group]
    missing = [column for column in columns if column not in test_df.columns]
    if missing:
        raise KeyError(f"[그룹{group}] 평가 데이터에 없는 피처: {missing}")
    # LSTM도 LightGBM과 같은 SHAP 선별 피처 순서를 강제한다.

    variant = AllFeaturesVariant(random_state=SEED)
    full_frame = variant.fit_transform(train_df[columns], y_full)
    test_frame = variant.transform(test_df[columns])
    # 전체 학습 중앙값으로 평가 결측을 채우고 컬럼 순서를 고정한다.
    scaler = MinMaxScaler().fit(full_frame)
    X_full_scaled = scaler.transform(full_frame)
    X_test_scaled = scaler.transform(test_frame)
    # 최종 스케일러도 라벨이 있는 전체 학습 기간에만 fit한다.

    version = model_name.split("_")[1]
    artifact = lstm_artifacts[(group, model_name)]
    best_epoch = int(artifact["pipeline"].best_epoch)
    lstm_params = {
        **MODEL_PARAMS,
        "epochs": max(1, best_epoch),
        "patience": max(2, best_epoch + 1),
    }
    # 검증에서 고른 epoch 수만큼 전체 학습하고 조기종료는 발동하지 않게 한다.
    if "loss_k" in artifact:
        lstm_params["loss_k"] = artifact["loss_k"]
        lstm_params["regression_weight"] = artifact["regression_weight"]
        # 카드 D(GWO)로 채택된 *_gwo 후보는 검증에서 찾은 loss_k/regression_weight를 그대로 재사용한다.

    final_model = make_lstm_pipeline(version, capacity_kw=capacity_kw, seq_len=SEQ_LEN, **lstm_params)
    final_model.fit(X_full_scaled, y_full)
    # 2022~2024년 전체 시퀀스로 최종 LSTM 가중치를 학습한다.
    X_with_context = np.vstack([X_full_scaled[-SEQ_LEN:], X_test_scaled])
    # 학습 마지막 24시간을 붙여 2025년 첫 시각의 과거 문맥을 제공한다.
    return final_model.predict(X_with_context)
    # 문맥 24행이 소비되므로 반환 길이는 평가 8,760행과 정확히 같다.


In [ ]:
def applyCalibrationIfAny(
    group: int,
    model_name: str,
    pred: np.ndarray,
    use_calibration: bool,
    test_hour_bucket: np.ndarray,
) -> np.ndarray:
    """검증 구간에서 고정한 보정 파라미터를 평가 구간 예측에 그대로 적용하는 함수

    Logic:
        - use_calibration이 False면 원본 예측을 그대로 반환한다
        - final_calibrator_kind가 "diurnal"이면 시각대별 (a,b), 아니면 전역 (a,b)를 적용한다
        - 두 경우 모두 fit()을 다시 호출하지 않고 Step 8에서 검증 구간에 적합해 둔 계수만 쓴다
    """
    if not use_calibration:
        return pred
    calibrator = final_calibrators[(group, model_name)]
    if final_calibrator_kind[(group, model_name)] == "diurnal":
        return calibrator.transform(pred, test_hour_bucket)
    return calibrator.transform(pred)


submission = loader["sample_submission"].copy()
# 원본 제출 양식을 복사해 행 순서와 필수 컬럼 순서를 보존한다.
submission["forecast_kst_dtm"] = pd.to_datetime(
    submission["forecast_kst_dtm"]
)
# 평가 피처와 같은 datetime 자료형으로 병합 키를 통일한다.

final_models_meta: dict[int, dict] = {}
# 그룹별 최종 채택 정보(final_choice)를 실행 중 검토할 수 있게 보관한다.
for group in GROUPS:
    target_col = f"kpx_group_{group}"
    train_df = datasets[group]["df"]
    test_df = prep_builder.load(group, "test")
    chosen = final_choice[group]
    # 이 그룹의 Step 8 최종 채택 정보(kind: raw/calibrated/ensemble)

    test_hour_bucket = pd.cut(
        test_df["target_hour_ldaps"].to_numpy(),
        bins=HOUR_BINS, labels=HOUR_LABELS, include_lowest=True,
    ).to_numpy()
    # 카드 B 시각대별 보정이 채택된 경우 평가 구간에도 동일한 구간 정의를 적용하기 위해 미리 만든다.

    if chosen["kind"] == "ensemble":
        pred_a = fitFinalCandidate(group, chosen["ensemble_a"], train_df, test_df)
        pred_b = fitFinalCandidate(group, chosen["ensemble_b"], train_df, test_df)
        pred_a = applyCalibrationIfAny(group, chosen["ensemble_a"], pred_a, chosen["a_calibrated"], test_hour_bucket)
        pred_b = applyCalibrationIfAny(group, chosen["ensemble_b"], pred_b, chosen["b_calibrated"], test_hour_bucket)
        pred = chosen["ensemble_w"] * pred_a + (1 - chosen["ensemble_w"]) * pred_b
        # 검증 구간에서 고정한 가중치를 그대로 재사용하고, 평가 구간에서 다시 탐색하지 않는다.
        columns_used = "ensemble"
    else:
        model_name = chosen["model_name"]
        pred = fitFinalCandidate(group, model_name, train_df, test_df)
        pred = applyCalibrationIfAny(
            group, model_name, pred, chosen["kind"] == "calibrated", test_hour_bucket,
        )
        columns_used = model_name

    if len(pred) != len(test_df):
        raise ValueError(
            f"[그룹{group}] 예측 길이 불일치: 예측 {len(pred)}, 평가 {len(test_df)}"
        )
        # 첫 24시간 누락을 0으로 숨기던 이전 제출 버그의 재발을 차단한다.
    final_models_meta[group] = chosen
    # 그룹별 최종 채택 정보를 보관한다.

    pred_series = pd.Series(
        pred,
        index=pd.to_datetime(test_df["forecast_kst_dtm"]),
    )
    submission[target_col] = pred_series.reindex(
        submission["forecast_kst_dtm"]
    ).to_numpy()
    # 예측 시각 인덱스로 sample_submission의 정확한 행 순서에 맞춰 넣는다.
    print(
        f"✅ [그룹{group}] {chosen['kind']} {columns_used} | "
        f"검증 총점 {chosen['score']:.4f} | 예측 {len(pred):,}시간"
    )
    # 그룹별 채택 방식·검증 총점·예측 길이를 실행 로그에 남긴다.


In [ ]:
# 제출 스키마·결측·물리 범위·첫 24시간 예측을 검증한다.
sample = loader["sample_submission"]
assert list(submission.columns) == list(sample.columns), "제출 컬럼 순서 불일치!"
assert len(submission) == len(sample) == 8_760, "제출 행 수가 8,760이 아닙니다!"
assert submission.isna().sum().sum() == 0, "제출 파일에 NaN이 있습니다!"
# 결측을 ffill/0으로 숨기지 않고 시각 정렬 실패를 그대로 중단한다.
for group in GROUPS:
    column = f"kpx_group_{group}"
    assert (submission[column] >= 0.0).all(), f"{column}에 음수 예측값이 있습니다!"
    assert (
        submission[column] <= RATED_CAPACITY_KW[group]
    ).all(), f"{column}에 설비용량 초과 예측값이 있습니다!"
    assert submission[column].nunique() > 1, f"{column} 예측이 상수입니다!"
    # 발전량은 0~그룹 설비용량(kWh/h) 범위이며 전체 기간이 상수로 무너지면 안 된다.

out_path = f"{ROOT}/submission_baseline8.csv"
submission.to_csv(out_path, index=False, encoding="utf-8-sig")
# 대회 제출 호환을 위해 UTF-8 BOM·인덱스 제외 형식으로 저장한다.
print(f"스키마 검증 통과 ✅")
print(f"제출 파일 저장 완료: {out_path}")
# 검증 성공과 최종 파일 경로를 명시한다.
display(submission.head())
# 2025년 첫 시각부터 세 그룹 예측이 채워졌는지 확인한다.


In [ ]:
# 최종 제출 예측이 입력에 반응하는지 그룹별 분포를 확인한다.
for group in GROUPS:
    column = f"kpx_group_{group}"
    series = submission[column]
    print(
        f"[{column}] 고유값 {series.nunique():,}개 | "
        f"평균 {series.mean():,.0f} | 최소 {series.min():,.0f} | "
        f"최대 {series.max():,.0f} kWh"
    )
    # 고유값이 충분하고 0~설비용량 범위면 상수 예측 붕괴 가능성이 낮다.

submission.set_index("forecast_kst_dtm")[
    [f"kpx_group_{group}" for group in GROUPS]
].head(24 * 14).plot(
    figsize=(12, 4),
    title="Baseline 8 제출 예측 — 2025년 첫 2주",
)
plt.ylabel("예측 발전량 (kWh)")
plt.xlabel("예보 대상 시각")
plt.tight_layout()
plt.show()
# 첫 2주 시간 변화와 그룹별 차이를 마지막으로 시각 검토한다.

print("=== 그룹별 최종 채택 요약 (카드 F) ===")
for group in GROUPS:
    chosen = final_models_meta[group]
    gap = chosen["score"] - BASELINE7_SCORE[group]
    print(f"[그룹{group}] {chosen['kind']:<10} baseline_7 대비 {gap:+.4f} (검증 총점 {chosen['score']:.4f})")
mean_score = float(np.mean([final_models_meta[g]["score"] for g in GROUPS]))
mean_baseline = float(np.mean(list(BASELINE7_SCORE.values())))
print(f"3그룹 평균 검증 총점: {mean_score:.4f} (baseline_7 {mean_baseline:.4f} 대비 {mean_score - mean_baseline:+.4f})")
# baseline7_howto_next.md §1-2 Phase 3 목표(0.6672 → 0.68 잠정치)와 비교할 수 있는 최종 요약


---
## 부록. VMD 입력 피처 분해 + mRMR 기반 IMF 특징 선택 (카드 G, 선택적 탐색 스파이크)

`ws10`(LDAPS IDW 대표 풍속)·`gfs_ws_hub`(GFS 허브높이 외삽 풍속)를 VMD로 K개 모드로 분해해
추세·고주파 성분을 파생 피처로 추가하고, mRMR로 사전 축소한 뒤 기존 `LGBM_selected` 파이프라인
(Step 2)에 편입하는 **선택적** 실험이다(`baseline7_idea_evaluation.md` 아이디어 1, 5). 기본은
`ENABLE_VMD = False`로 꺼져 있어 평소 실행에는 영향이 없다.

**절대 하지 말 것**: 타깃(발전량)을 VMD로 분해해 자기회귀 예측하지 않는다(day-ahead 일괄 제출
구조상 제출 불가능 — `Persistence_oracle_lag1`과 동일한 결함). 학습+검증 구간을 합쳐 한 번에
VMD를 돌리지 않는다(검증 구간 스펙트럼 정보 누수).

**1차 게이트**: 그룹1에서 `LGBM_selected` 대비 검증 총점이 **+0.005 이상** 개선되어야만
게이트를 통과한 것으로 보고, 그 이상(그룹 2·3 확장, 실제 Step 2 파이프라인 편입)을 진행한다.
게이트를 통과해도 **이 스파이크는 자동으로 Step 2에 반영되지 않는다** — 셀 출력의 채택된
IMF 컬럼 목록을 확인한 뒤, 사람이 Step 1의 `datasets[group]["df"]`에 병합하고 Step 2부터
다시 실행해야 한다(신규 의존성·구조 변경을 조용히 자동 반영하지 않는다는 저장소 원칙).


In [ ]:
ENABLE_VMD = False
# True로 바꾸면 카드 G(VMD+mRMR) 스파이크가 실행된다. 기본은 False — 원안 파이프라인 우선.
VMD_TARGET_COLS = ["ws10", "gfs_ws_hub"]
# 타깃(발전량)이 아니라 입력 예보 풍속만 분해한다.
VMD_K_GRID = [4, 6, 8]
VMD_ALPHA_GRID = [1_000, 2_000]
# 재구성 오차가 아니라 검증 총점 기준으로 조합을 고른다(baseline7_idea_evaluation.md 1-2절).
VMD_REFIT_STRIDE_HOURS = 168
# 검증 구간은 매 시점이 아니라 주 단위로만 재적합해 연산 비용을 통제한다.
MRMR_TOP_K = 6
# 그룹당 최대 16개 IMF를 mRMR로 6개까지 줄여 SHAP 선별의 "집단 탈락"을 막는다.
GATE_GROUP = 1
GATE_IMPROVEMENT = 0.005
# 1차 게이트 그룹과 최소 개선폭. 이 값 미만이면 그룹 2·3으로 확장하지 않는다.

try:
    from vmdpy import VMD
    from sklearn.feature_selection import mutual_info_regression
    VMD_AVAILABLE = True
except ImportError:
    VMD_AVAILABLE = False
    print(
        "⚠️ vmdpy가 설치되어 있지 않습니다. `pip install vmdpy`로 설치하거나 "
        "ENABLE_VMD=False로 두고 이 스파이크를 건너뜁니다."
    )
    # 이 저장소는 신규 의존성을 자동 설치하지 않는다 — 설치 여부는 사용자가 결정한다.


In [ ]:
def causalVmdDecompose(
    signal: np.ndarray,
    k: int,
    alpha: float,
    train_size: int,
    refit_stride: int = VMD_REFIT_STRIDE_HOURS,
) -> np.ndarray:
    """학습 구간 1회 적합 + 검증 구간 주 단위 확장 윈도우 재적합으로 VMD 모드를 만드는 함수

    Args:
        - signal: 그룹별 시간순 입력 풍속 배열(ws10 또는 gfs_ws_hub), 학습+검증 구간 전체
        - k: 분해할 모드(IMF, 시계열을 나눈 한 조각) 개수
        - alpha: 대역폭 제약 파라미터(클수록 모드가 매끈해짐)
        - train_size: 학습 구간 길이(splitByTime과 동일한 경계, 뒤는 검증 구간)
        - refit_stride: 검증 구간 재적합 주기(시간). 기본 168시간(1주)

    Returns:
        - shape (k, len(signal))인 모드 배열. 각 시점은 그 시점까지 알려진 신호만으로 계산된
          모드값을 담는다(검증 구간의 스펙트럼 정보가 학습 구간 모드에 섞이지 않는다)

    Logic:
        - 학습 구간(0~train_size)은 그 구간 신호만으로 한 번 분해한다.
        - 검증 구간은 매 재적합 시점마다 "지금까지 알려진 전체 신호"로 다시 분해하고,
          방금 분해한 윈도우의 오른쪽 끝(가장 최근 refit_stride개) 값만 채택한다.
          윈도우 오른쪽 끝은 VMD 경계 왜곡이 가장 큰 지점이므로 완화되지 않는 구조적 한계를
          그대로 안고 간다(baseline7_idea_evaluation.md 1-1절 (3)).
    """
    n = len(signal)
    modes = np.full((k, n), np.nan)
    tau, dc, init, tol = 0.0, 0, 1, 1e-7
    # tau=0(잡음 없음 가정)·DC=0(직류 모드 강제 없음)·init=1(균등 초기화)은 vmdpy 표준값

    u_train, _, _ = VMD(signal[:train_size], alpha, tau, k, dc, init, tol)
    modes[:, :train_size] = u_train[:, :train_size]
    # 학습 구간은 그 구간만으로 딱 한 번 적합한다 — 검증 구간 정보가 전혀 섞이지 않는다.

    pos = train_size
    while pos < n:
        end = min(pos + refit_stride, n)
        window = signal[:end]
        # 지금까지(현재 재적합 시점까지) 알려진 신호 전체로 다시 분해한다(확장 윈도우).
        u_win, _, _ = VMD(window, alpha, tau, k, dc, init, tol)
        chunk_len = end - pos
        modes[:, pos:end] = u_win[:, -chunk_len:]
        # 방금 만든 윈도우의 가장 최근 구간만 이번 재적합 주기의 값으로 채택한다.
        pos = end
    return modes


def selectImfFeaturesMrmr(
    imf_df: pd.DataFrame,
    y_train: np.ndarray,
    top_k: int = MRMR_TOP_K,
    seed: int = SEED,
) -> list[str]:
    """VMD의 IMF를 mRMR(관련성은 높고 중복은 낮게)로 고르는 함수

    Args:
        - imf_df: 학습 구간의 IMF 컬럼만 담은 DataFrame(vmd_ws10_mode1..K 등)
        - y_train: 학습 구간 타깃(발전량 kWh)
        - top_k: 최종적으로 남길 IMF 개수

    Logic:
        - relevance: mutual_info_regression(imf, y_train) — 비선형 관계도 포착
        - redundancy: 이미 선택된 IMF들과의 평균 절대 상관계수
        - 매 스텝 (relevance - redundancy)가 최대인 컬럼을 그리디하게 추가한다
        - 반드시 학습 구간(imf_df, y_train)에서만 계산한다. 선택된 컬럼 이름 집합은
          검증 구간에도 고정 적용하고, 재적합 윈도우가 바뀌어도 다시 뽑지 않는다
          (baseline7_idea_evaluation.md 5-3절 누수 방지 규칙)
    """
    relevance = pd.Series(
        mutual_info_regression(imf_df, y_train, random_state=seed),
        index=imf_df.columns,
    )
    if relevance.std() < 0.05 * max(relevance.mean(), 1e-9):
        print(
            "⚠️ mRMR relevance가 비정상적으로 균일합니다 — PCC-GRA 폴백을 검토하세요 "
            "(baseline7_idea_evaluation.md 5-2절)."
        )
        # 자동으로 PCC-GRA를 실행하지는 않는다. 이 저장소는 신규 로직을 조용히 대체하지 않는다.

    selected: list[str] = [relevance.idxmax()]
    remaining = [c for c in imf_df.columns if c not in selected]
    while len(selected) < min(top_k, len(imf_df.columns)) and remaining:
        redundancy = {
            c: imf_df[selected].corrwith(imf_df[c]).abs().mean() for c in remaining
        }
        mrmr_score = {c: relevance[c] - redundancy[c] for c in remaining}
        best = max(mrmr_score, key=mrmr_score.get)
        selected.append(best)
        remaining.remove(best)
    return selected
    # 선택된 IMF 컬럼명 리스트. 게이트를 통과하면 datasets[group]["df"]에 이 컬럼만 병합한다


In [ ]:
vmd_gate_results: dict[int, dict] = {}
# 그룹별 "VMD+mRMR 추가 전/후" LGBM_selected 검증 총점을 게이트 판단용으로 담는다.

if ENABLE_VMD and VMD_AVAILABLE:
    baseline_lookup = {
        (row["group"], row["model_name"]): row["score"] for row in records
    }
    # Step 2에서 이미 계산된 LGBM_selected 검증 총점을 그대로 기준선으로 재사용한다.
    vmd_target_groups = [GATE_GROUP]
    # 1차 게이트를 통과하기 전에는 그룹1만 시험한다(카드 G 1차 게이트, 그룹 2·3은 게이트 후 확장).

    for group in vmd_target_groups:
        df_group = datasets[group]["df"]
        capacity_kw = RATED_CAPACITY_KW[group]
        base_cols = selected_features[group]
        train_size = len(datasets[group]["X_train"])
        # splitByTime과 동일한 학습·검증 경계를 그대로 쓴다.
        baseline_score = baseline_lookup.get((group, "LGBM_selected"))
        actual_aligned = datasets[group]["y_valid"][SEQ_LEN:]

        best_score, best_cols, best_k, best_alpha = -np.inf, [], None, None
        for k in VMD_K_GRID:
            for alpha in VMD_ALPHA_GRID:
                imf_train_parts, imf_valid_parts = [], []
                for target_col in VMD_TARGET_COLS:
                    if target_col not in df_group.columns:
                        continue
                    signal = df_group[target_col].to_numpy(dtype=float)
                    modes = causalVmdDecompose(signal, k=k, alpha=alpha, train_size=train_size)
                    cols = [f"vmd_{target_col}_mode{i + 1}" for i in range(k)]
                    imf_train_parts.append(pd.DataFrame(modes[:, :train_size].T, columns=cols))
                    imf_valid_parts.append(pd.DataFrame(modes[:, train_size:].T, columns=cols))
                if not imf_train_parts:
                    continue
                imf_train_df = pd.concat(imf_train_parts, axis=1)
                imf_valid_df = pd.concat(imf_valid_parts, axis=1)
                # 이 K×alpha 조합의 IMF를 만들고, mRMR로 학습 구간에서만 축소한다.

                selected_imf_cols = selectImfFeaturesMrmr(
                    imf_train_df, datasets[group]["y_train"], top_k=MRMR_TOP_K,
                )
                trial_X_train = pd.concat(
                    [
                        datasets[group]["X_train"][base_cols].reset_index(drop=True),
                        imf_train_df[selected_imf_cols],
                    ],
                    axis=1,
                )
                trial_X_valid = pd.concat(
                    [
                        datasets[group]["X_valid"][base_cols].reset_index(drop=True),
                        imf_valid_df[selected_imf_cols],
                    ],
                    axis=1,
                )
                trial_model = trainLightGBM(
                    trial_X_train, datasets[group]["y_train"],
                    trial_X_valid, datasets[group]["y_valid"], capacity_kw,
                )
                trial_pred = predictKw(trial_model, trial_X_valid, capacity_kw)[SEQ_LEN:]
                trial_score = metrics.summarize(trial_pred, actual_aligned, group)["score"]
                # 재구성 오차가 아니라 검증 총점으로만 K·alpha 조합을 고른다.
                if trial_score > best_score:
                    best_score, best_cols = trial_score, selected_imf_cols
                    best_k, best_alpha = k, alpha

        improvement = (best_score - baseline_score) if baseline_score is not None else np.nan
        vmd_gate_results[group] = {
            "group": group,
            "baseline_score": baseline_score,
            "best_score_with_vmd": best_score,
            "improvement": improvement,
            "gate_passed": bool(baseline_score is not None and improvement >= GATE_IMPROVEMENT),
            "best_k": best_k,
            "best_alpha": best_alpha,
            "selected_imf_cols": best_cols,
        }

    vmd_gate_df = pd.DataFrame(vmd_gate_results.values())
    display(vmd_gate_df[["group", "baseline_score", "best_score_with_vmd", "improvement", "gate_passed", "best_k", "best_alpha"]])
    if any(row["gate_passed"] for row in vmd_gate_results.values()):
        print("✅ 1차 게이트 통과 — 그룹 2·3으로 확장을 검토하세요. 채택하려면 위 selected_imf_cols를")
        print("   Step 1의 datasets[group][\"df\"]에 병합한 뒤 Step 2부터 다시 실행해야 합니다.")
    else:
        print("⛔ 1차 게이트 미달 — 카드 G는 여기서 종료하고 원안(LGBM_selected)을 유지합니다.")
else:
    print("ℹ️ ENABLE_VMD=False 또는 vmdpy 미설치 — 카드 G 스파이크를 건너뜁니다.")


---
## 정리

| 카드 | Baseline 8 적용 |
|---|---|
| 카드 A | `predictions_store` + `summarizeErrorBand`/`recommendAction`으로 그룹·모델별 오차 밴드 진단 |
| 카드 B | `FicrCalibrator`(전역) + `DiurnalFicrCalibrator`(시각대별, 불균일 그룹만) 사후 보정 |
| 카드 C | `makeFicrObjective` + `trainFicrLightGBM` 2단계 웜업으로 `LGBM_ficr` 후보 추가 |
| 카드 D | 순수 numpy `GreyWolfOptimizer`로 LSTM·`LGBM_ficr`의 `loss_k`/`regression_weight`(`k`) 재탐색 |
| 카드 E | `target_hour_ldaps` 3구간(단기/중기/장기) 오차 밴드 불균일 진단 |
| 카드 F | 모든 후보(원본/보정/GWO)를 그룹별로 비교하고, 기준선을 넘는 후보가 2개 이상이면 앙상블까지 시도해 최종 채택 |
| 카드 G(선택) | VMD+mRMR 피처 스파이크. 기본 `ENABLE_VMD=False`로 비활성, 게이트 통과 시에도 수동 반영 필요 |
| 제출 | 그룹별 채택 방식(원본/보정/앙상블)을 그대로 반영해 전체 재학습, `submission_baseline8.csv` 저장 |

실제 채택 모델과 점수는 Step 4·7·8의 실행 결과를 기준으로 판단한다. `BASELINE7_SCORE`
(그룹1 `0.6555` / 그룹2 `0.6877` / 그룹3 `0.6583`)는 `baseline_7_results.csv` 실측값이며,
이 노트북의 모든 카드 B~F 채택 여부는 이 기준선과의 비교로 결정된다.
